In [6]:
import torch
print("GPU aktif mi?", torch.cuda.is_available())
print("Cihaz:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU aktif mi? True
Cihaz: Tesla T4


# GoEmotions — Tam araştırma notebook’u (deneyler önce, teslim en son)

**ADIM 0–11:** araştırma ve ölçüm. **ADIM 12:** hocanın orijinal PDF formu ve 5 slaytlık sunum. **ADIM 13:** ZIP ve teslim.

Teslim adımları varsayılan olarak kapalıdır (`GENERATE_DELIVERABLES = False`). `Run all` deneyleri çalıştırır, teslim dosyalarını oluşturmaz. Sonuçlar hazır olduğunda teslim adımlarını etkinleştir.


### Kim neyi yaptı — bu notebook'ta nerede?

| Bölüm | Ne | Kim |
|---|---|---|
| ADIM 0–4.5 | Veri, yazım hatası üreticisi, TF-IDF + LR baseline, 3 yazım hatası çekilişi | Rümeysa (veri, üretici), takım |
| ADIM 4.6 | Yazım hatası testleri, veri sızıntısı ve etiket dağılımı kontrolü | Tarık'ın görevi |
| ADIM 4.7 | Baseline hata analizi (slayt 5) | Fatma |
| ADIM 4.8 | Spell-check pilotu | Rümeysa |
| ADIM 5–9 | DistilBERT ön denemesi (eğitim seed'i sabit değil; sonuç tablosunda kullanılmıyor) | Lara |
| ADIM 5.3 | DistilBERT 3 eğitim seed'i + %5/%10/%20 ablasyonu (sonuç tablosu) | Fatma |


In [7]:
import random
import numpy as np
import torch
import json, time, math

## ADIM 0 — Kurulum, proje kimliği ve tekrar üretilebilirlik

**Neden?** Kütüphaneleri kuruyoruz, sabit tohumlar tanımlıyoruz, çıktıları tek klasörde saklıyoruz. **Form:** Kutu 4, 8 ve 9. Aşağıdaki takım alanlarını doldur; **gerçek olmayan isim/numara eklenmedi**.

**This notebook includes the original instructor PDF form (embedded); it generates a completed copy of the form, not a redesigned proposal.**

In [8]:
%pip -q install datasets
from pathlib import Path
import numpy as np
import pandas as pd
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
GENERATE_DELIVERABLES = False  # Deneyler bitene kadar False; en son True yap
TEAM_NUMBER = "FLR44_PROPOSAL"  # Buraya gerçek takım numarası
TEAM_NAME = "GoEmotions Robustness Study"
MEMBERS = ["[Graduate research lead: Name / Student ID / CSE 654]", "[Graduate member: Name / Student ID / CSE 654]", "[Undergraduate member: Name / Student ID / CSE 484]"]
AI_TOOLS_USED = "ChatGPT (OpenAI), Claude (Anthropic)"  # Yalnızca gerçekten kullanılan araçlar
TEAM_ROLES = "Graduate lead: literature, design; second graduate: modelling and robustness; undergraduate: data, evaluation/demo"
OUTPUT_DIR = Path('/content/goemotions_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Cihaz:', DEVICE, '| CUDA:', torch.cuda.is_available())
print('Çıktı klasörü:', OUTPUT_DIR)

Cihaz: cuda | CUDA: True
Çıktı klasörü: /content/goemotions_outputs


## ADIM 1 — GoEmotions Veri Setinin Yüklenmesi ve Tanınması

### Amaç
Bu aşamada, araştırmamızda kullanacağımız GoEmotions veri setini Google Colab ortamına yüklüyor ve veri setinin temel özelliklerini inceliyoruz.

### Neden yapıyoruz?
Araştırma sorumuz, yazım hatalarının duygu sınıflandırma modellerinin başarısını nasıl etkilediğidir.

Bu soruyu cevaplayabilmek için duygu etiketleri önceden belirlenmiş gerçek metinlere ihtiyacımız var. GoEmotions, Reddit yorumlarından oluşan ve 27 duygu kategorisi ile 1 nötr etiketi içeren bir veri setidir.

Bu veri seti sayesinde modellerimizin tahminlerini önceden belirlenmiş duygu etiketleriyle karşılaştırabileceğiz.

### Kodun yaptığı işlemler
1. `datasets` kütüphanesini kurar.
2. GoEmotions veri setinin `simplified` sürümünü Hugging Face üzerinden yükler.
3. Eğitim, doğrulama ve test bölümlerindeki örnek sayılarını gösterir.
4. Veri setindeki 28 duygu etiketini listeler.
5. Eğitim kümesinden ilk üç gerçek yorumu ve bunların duygu etiketlerini gösterir.

### Veri setinin yapısı

| Veri bölümü | Örnek sayısı | Kullanım amacı |
|---|---:|---|
| Train (Eğitim) | 43.410 | Modelleri eğitmek |
| Validation (Doğrulama) | 5.426 | Model ayarlarını belirlemek |
| Test | 5.427 | Modellerin başarısını değerlendirmek |
| **Toplam** | **54.263** | |

Her kayıt üç alan içerir:
- `text`: Yorumun metni
- `labels`: Yorumun duygu etiketleri
- `id`: Yoruma ait kimlik

GoEmotions çok etiketli bir veri setidir. Yani bir yorum aynı anda birden fazla duygu etiketi taşıyabilir.

### Kaynak ve lisans
- Veri seti: https://huggingface.co/datasets/google-research-datasets/go_emotions
- Orijinal çalışma: https://github.com/google-research/google-research/tree/master/goemotions
- Lisans: Apache License 2.0

### Gerçek veri örneği
**Metin:** My favourite food is anything I didn't have to cook myself.

**Etiket:** `neutral` (nötr)

Bu örnek doğrudan veri setinin eğitim bölümünden alınmıştır.

### Elde edilen sonuç
GoEmotions veri seti başarıyla yüklenmiş, toplam 54.263 yorum ve 28 etiket bulunduğu doğrulanmıştır.

Bu aşamada model eğitimi yapılmamıştır. Yalnızca verinin yapısı ve içeriği incelenmiştir.

### Proje önerisiyle ilişkisi
**Kutu 3 — Data:** Kullanılacak veri setinin adı, içeriği, eğitim/doğrulama/test sayıları ve gerçek örneği belirlenmiştir. Veri kaynağı ve lisans bilgisi aşağıda belirtilmiştir; proje önerisine de aktarılacaktır.

**Veri kaynağı:** https://huggingface.co/datasets/google-research-datasets/go_emotions (`simplified`) · **Lisans:** Apache-2.0 · **Dil:** İngilizce. Orijinal makale: Demszky et al. (2020), https://arxiv.org/abs/2005.00547.

In [9]:
!pip install -q datasets

from datasets import load_dataset

# GoEmotions'ı Hugging Face'ten indir
ds = load_dataset("google-research-datasets/go_emotions", "simplified")

# 1) Kaç örnek var?
print(ds)

# 2) Duygu isimleri neler? Veri Setinin yapısı.
label_names = ds["train"].features["labels"].feature.names
print("Duygu sayısı:", len(label_names))
print(label_names)

# 3) İlk 3 örneğe bak
for i in range(3):
    ornek = ds["train"][i]
    print("\nMetin:", ornek["text"])
    print("Duygular:", [label_names[j] for j in ornek["labels"]])

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
Duygu sayısı: 28
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

Metin: My favourite food is anything I didn't have to cook myself.
Duygular: ['neutral']

Metin: Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actually dead
Duygular: ['neutral']

Metin: WHY THE FUCK IS BAYLESS ISOING
Duygular: ['anger']


### Gerçek duygu örneği ve etiket sıklıkları

**Neden?** Formun veri bölümünde nötr olmayan, gerçekten veri setinde bulunan bir örnek sunmak; nadir duyguları incelemek. Veriler yalnızca eğitim kümesinden seçilir.

In [10]:
# ADIM 1 — Duygu etiketlerinin dağılımı ve gerçek veri örneği

from collections import Counter
import numpy as np
import pandas as pd

# Eğitim verisindeki etiketleri al
train_labels = ds["train"].with_format(None)["labels"]

# Her duygu etiketinin kaç kez kullanıldığını hesapla
counts = Counter(
    label
    for labels in train_labels
    for label in labels
)

label_counts = np.array(
    [counts.get(i, 0) for i in range(len(label_names))],
    dtype=int
)

# Etiket dağılımını tablo olarak göster
label_df = pd.DataFrame({
    "emotion": label_names,
    "train_count": label_counts
}).sort_values("train_count")

print(label_df.to_string(index=False))

# Hocanın proposal formu için gerçek bir örnek seç
REAL_EXAMPLE_TEXT = None
REAL_EXAMPLE_LABEL = None

for row in ds["train"].with_format(None):
    if (
        len(row["labels"]) == 1
        and label_names[row["labels"][0]] in ["gratitude", "joy", "sadness", "anger"]
        and 15 <= len(row["text"]) <= 120
    ):
        REAL_EXAMPLE_TEXT = row["text"]
        REAL_EXAMPLE_LABEL = label_names[row["labels"][0]]

        print("\nGerçek eğitim örneği:")
        print("Metin:", repr(REAL_EXAMPLE_TEXT))
        print("Etiket:", REAL_EXAMPLE_LABEL)
        break

if REAL_EXAMPLE_TEXT is None:
    print("Belirtilen koşullara uygun örnek bulunamadı.")


       emotion  train_count
         grief           77
         pride          111
        relief          153
   nervousness          164
 embarrassment          303
       remorse          545
          fear          596
        desire          641
       disgust          793
    excitement          853
      surprise         1060
        caring         1087
   realization         1110
disappointment         1269
       sadness         1326
     confusion         1368
           joy         1452
         anger         1567
      optimism         1581
   disapproval         2022
          love         2086
     curiosity         2191
     amusement         2328
     annoyance         2470
     gratitude         2662
      approval         2939
    admiration         4130
       neutral        14219

Gerçek eğitim örneği:
Metin: 'WHY THE FUCK IS BAYLESS ISOING'
Etiket: anger


## ADIM 2 — Araştırma Sorusunun Belirlenmesi

### Amaç
GoEmotions veri setini kullanarak cevaplamak istediğimiz araştırma sorusunu belirlemek.

### Araştırma sorumuz

**İngilizce:**

Does a Transformer-based emotion classifier exhibit a smaller macro-F1 performance drop than a TF-IDF + Logistic Regression baseline when synthetic spelling errors are introduced into 10% of eligible words in the GoEmotions test set?

**Türkçe:**

GoEmotions test veri setindeki uygun kelimelere %10 olasılıkla yapay yazım hatası eklendiğinde, Transformer tabanlı duygu sınıflandırma modelinin Macro-F1 başarı kaybı, TF-IDF + Logistic Regression modelinden daha mı az olur?

### Araştırmanın amacı
Yazım hatalarının duygu sınıflandırma performansına etkisini ölçmek ve iki farklı model yaklaşımının bu hatalara karşı dayanıklılığını karşılaştırmak.

### Araştırma hipotezi
Transformer tabanlı modelin yazım hataları karşısında daha az performans kaybı yaşayacağını öngörüyoruz.

Bu bir varsayımdır; deney sonuçları hipotezimizi desteklemeyebilir.

### Hipotezin desteklenmediği ("hayır") durum
Transformer modelinin Macro-F1 kaybı baseline modelinden daha düşük değilse veya iki modelin kayıpları arasındaki fark için hesaplanan %95 eşleştirilmiş bootstrap güven aralığı sıfırı içeriyorsa, Transformer modelinin daha dayanıklı olduğu sonucuna varamayız. Bu istatistiksel değerlendirme deneyin sonraki aşamasında yapılacaktır.

### Deney planı
1. Her iki modeli orijinal eğitim verisi üzerinde eğitmek.
2. Modelleri orijinal test yorumlarında değerlendirmek.
3. Aynı test yorumlarına kontrollü yazım hataları eklemek.
4. Modelleri yazım hatalı yorumlarda yeniden değerlendirmek.
5. Macro-F1 başarı puanlarındaki düşüşleri karşılaştırmak.

### Proje önerisiyle ilişkisi
**Kutu 1 — Research Question:** Araştırma sorusunun ve karşılaştırılacak yöntemlerin belirlenmesi.

**Kutu 5 — Evaluation:** Başarı düşüşünün Macro-F1 üzerinden değerlendirilmesi planı.

Araştırma sorusu güncel literatür incelemesinden sonra son hâline getirilecektir.

### Önceden belirlenmiş “hayır” sonucu
Transformer modelinin Macro-F1 kaybı baseline modelinden küçük değilse **veya** (baseline kaybı − Transformer kaybı) farkının eşleştirilmiş bootstrap %95 güven aralığı sıfırı içeriyorsa, daha dayanıklı olduğu sonucuna varamayız. Test verisi yalnızca final değerlendirme için kullanılır.

## ADIM 3 — Yapay Yazım Hataları Oluşturma (Synthetic Typo Generation)

### Amaç
Bu aşamada, GoEmotions veri setindeki yorumların yazım hataları içeren alternatif sürümlerini oluşturuyoruz. Amacımız, yazım hatalarının duygu sınıflandırma modellerinin tahmin başarısını nasıl etkilediğini araştırmak.

### Neden yapıyoruz?
Araştırmamızda iki farklı modeli karşılaştıracağız:

- **TF-IDF + Logistic Regression:** Temel karşılaştırma modelimiz (baseline).
- **Transformer tabanlı model:** Daha gelişmiş karşılaştırma modelimiz.

Her iki modeli de orijinal eğitim verileriyle eğiteceğiz. Ardından aynı test yorumlarını hem orijinal hem de yazım hatası eklenmiş hâlleriyle modellere vereceğiz.

Böylece yazım hataları nedeniyle modellerin Macro-F1 başarı puanlarında oluşan düşüşü karşılaştırabileceğiz.

### Kullanılan yöntem
- **Veri:** GoEmotions test kümesi (5.427 yorum).
- **Hedef hata oranı:** Uygun kelime başına %10 değişiklik olasılığı.
- **Uygun kelimeler:** En az 4 İngilizce harften oluşan kelimeler.
- **Hata türleri:**
  1. Harf silme (`happy` → `hapy`)
  2. Harf tekrarlama (`happy` → `happpy`)
  3. Komşu harflerin yerini değiştirme (`happy` → `hpapy`)
- **Özel işaretlerin korunması:** `[NAME]`, `[RELIGION]` gibi köşeli parantezli ifadeler değiştirilmez.
- **Tekrarlanabilirlik:** Sabit rastgelelik tohumları (`seed`) kullanılır.
- **Duygu etiketleri:** Orijinal etiketler değiştirilmeden korunur.

### Elde edilen gerçek sonuçlar

| Ölçüm | Sonuç |
|---|---:|
| Toplam test yorumu | 5.427 |
| Değiştirilmeye uygun kelime | 37.661 |
| Değiştirilen kelime | 3.719 |
| Gerçek değişiklik oranı | **%9,87** |
| En az bir kelimesi değişen yorum | 2.587 |
| Hiç değişmeyen yorum | 2.840 |

### Özel işaret kontrolü
Kod, özel işaretlerin korunmasını ayrıca test etmiştir.

**Orijinal:**

`I really appreciate [NAME] and respect [RELIGION]. This is wonderful!`

**Yazım hatalı:**

`I raelly apprciate [NAME] and respet [RELIGION]. Tis is wodnerful!`

**Sonuç:** `[NAME]` ve `[RELIGION]` ifadeleri değiştirilmeden korunmuştur.

### İlk değerlendirme
Hedeflenen %10 değişiklik olasılığına karşılık, uygun kelimelerin **%9,87'si** değiştirilmiştir.

5.427 test yorumunun 2.587'sinde (yaklaşık %47,7) en az bir kelime değiştirilmiş, 2.840 yorumda ise herhangi bir değişiklik yapılmamıştır.

Bu aşamada herhangi bir duygu sınıflandırma modeli eğitilmemiş veya değerlendirilmemiştir. Yalnızca yazım hatalı test metinlerini oluşturma yöntemi kontrol edilmiştir.

Yapay yazım hatalarının her durumda metnin anlamını koruduğu garanti edilmediğinden, bu durum deneyin sınırlılığı olarak değerlendirilecektir.

### Proje önerisiyle ilişkisi
**Kutu 4 — Method and Baseline:** Yazım hatalarının nasıl üretileceğini ve kontrollü deney koşullarını açıklar.

**Kutu 5 — Evaluation:** Orijinal ve yazım hatalı test yorumlarında Macro-F1 başarılarının karşılaştırılmasına temel oluşturur.

**Kutu 6 — First Look:** Veri üzerinde gerçekleştirilen ilk deneysel işlemin sayısal çıktısını sağlar. Ancak model başarısına ilişkin sonuç yerine geçmez; bu bölüm için ayrıca baseline modelinin performansı hesaplanacaktır.

In [11]:
# ADIM 3 — Yapay yazım hataları oluşturma
# [NAME], [RELIGION] gibi özel işaretler korunur.

import random
import re

def yazim_hatasi_ekle(metin, oran=0.10, seed=42):
    rng = random.Random(seed)

    # Köşeli parantezli ifadeleri tek parça olarak ayır.
    # Diğer metinlerde kelimeleri ve noktalama işaretlerini koru.
    parcalar = re.findall(
        r"\[[^\[\]\n]*\]|[A-Za-z]+|[^A-Za-z\[\]]+|[\[\]]",
        metin
    )

    degisen = 0
    uygun = 0

    for j, kelime in enumerate(parcalar):

        # [NAME], [RELIGION] vb. özel işaretleri değiştirme
        if kelime.startswith("[") and kelime.endswith("]"):
            continue

        # En az 4 İngilizce harften oluşan kelimeleri seç
        if not re.fullmatch(r"[A-Za-z]{4,}", kelime):
            continue

        uygun += 1

        # Her uygun kelimeyi %10 olasılıkla değiştir
        if rng.random() >= oran:
            continue

        i = rng.randint(1, len(kelime) - 2)
        tur = rng.choice(["sil", "tekrar", "yer_degistir"])

        if tur == "sil":
            yeni = kelime[:i] + kelime[i+1:]

        elif tur == "tekrar":
            yeni = kelime[:i] + kelime[i] + kelime[i:]

        else:
            yeni = (
                kelime[:i] + kelime[i+1]
                + kelime[i] + kelime[i+2:]
            )

        if yeni != kelime:
            parcalar[j] = yeni
            degisen += 1

    return "".join(parcalar), degisen, uygun


# 1 — Özel işaretlerin korunup korunmadığını kontrol et
kontrol_metni = (
    "I really appreciate [NAME] and respect [RELIGION]. "
    "This is wonderful!"
)

kontrol_sonucu, _, _ = yazim_hatasi_ekle(
    kontrol_metni, oran=1.0, seed=42
)

print("ÖZEL İŞARET KONTROLÜ")
print("Orijinal:", kontrol_metni)
print("Değişmiş:", kontrol_sonucu)

assert "[NAME]" in kontrol_sonucu
assert "[RELIGION]" in kontrol_sonucu
print("Özel işaretler korundu.")


# 2 — Gerçek test yorumlarında örnekleri göster
test = ds["test"]

print("\nGERÇEK TEST ÖRNEKLERİ")

for idx in [0, 5, 12, 20, 33]:
    orijinal = test[idx]["text"]

    bozuk, degisen, uygun = yazim_hatasi_ekle(
        orijinal, oran=0.10, seed=idx
    )

    duygular = [label_names[j] for j in test[idx]["labels"]]

    print(f"\nDuygu    : {duygular}")
    print(f"Orijinal : {orijinal}")
    print(f"Bozulmuş : {bozuk}")
    print(f"Değişen  : {degisen}/{uygun} uygun kelime")


# 3 — Tüm test setinin değişiklik oranını hesapla
toplam_uygun = 0
toplam_degisen = 0
degisen_yorum = 0

for i, metin in enumerate(test["text"]):
    bozuk, degisen, uygun = yazim_hatasi_ekle(
        metin, oran=0.10, seed=i
    )

    toplam_uygun += uygun
    toplam_degisen += degisen

    if degisen > 0:
        degisen_yorum += 1

print("\nTEST SETİ ANALİZİ")
print("Toplam yorum:", len(test))
print("Uygun kelime:", toplam_uygun)
print("Değiştirilen kelime:", toplam_degisen)
print(
    "Gerçek değişiklik oranı:",
    f"%{100 * toplam_degisen / toplam_uygun:.2f}"
    if toplam_uygun else "%0"
)
print("En az bir kelimesi değişen yorum:", degisen_yorum)

ÖZEL İŞARET KONTROLÜ
Orijinal: I really appreciate [NAME] and respect [RELIGION]. This is wonderful!
Değişmiş: I raelly apprciate [NAME] and respet [RELIGION]. Tis is wodnerful!
Özel işaretler korundu.

GERÇEK TEST ÖRNEKLERİ

Duygu    : ['sadness']
Orijinal : I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!
Bozulmuş : I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!
Değişen  : 0/11 uygun kelime

Duygu    : ['gratitude']
Orijinal : Thank you for asking questions and recognizing that there may be things that you don’t know or understand about police tactics. Seriously. Thank you.
Bozulmuş : Thank you for asking questions and recognizing that there may be thins that you don’t know or understand about police tactics. Seriously. Thhank you.
Değişen  : 2/15 uygun kelime

Duygu    : ['desire']
Orijinal : Translation }}} I wish I could afford it.
Bozulmuş : Translation }}} I wish I could affor

## ADIM 4 — Baseline Modelinin Kurulması (TF-IDF + Logistic Regression)

### Amaç
GoEmotions veri setindeki yorumların duygu etiketlerini tahmin eden ilk modelimizi oluşturmak.

### Neden yapıyoruz?
Araştırmamızda Transformer tabanlı modelin yazım hatalarına karşı daha dayanıklı olup olmadığını inceleyeceğiz. Bunun için karşılaştırabileceğimiz daha basit bir başlangıç modeline ihtiyacımız var. Bu modele **baseline** denir.

### Kullanılacak yöntem
- **TF-IDF:** Metinleri sayısal özelliklere dönüştürür.
- **Logistic Regression:** Duygu etiketlerini tahmin eder.
- **One-vs-Rest:** Her duygu için ayrı bir sınıflandırıcı kullanarak çok etiketli tahmin yapılmasını sağlar.

### Deney yaklaşımı
Model yalnızca orijinal eğitim verileriyle eğitilecektir. Doğrulama kümesiyle karar eşikleri belirlenecek; test kümesi ise nihai performansı ölçmek için kullanılacaktır.

Ardından model, aynı test yorumlarının yapay yazım hataları eklenmiş sürümlerinde değerlendirilecektir.

### Proje önerisiyle ilişkisi
**Kutu 4 — Method and Baseline:** Başlangıç modelinin tanımlanması.

**Kutu 5 — Evaluation:** Macro-F1 ile performansın ölçülmesi.

**Kutu 6 — First Look:** Çalıştırılan modelden elde edilen gerçek sayısal sonuçların raporlanması.

Önce bütün modeli tek seferde çalıştırmak yerine, eğitim verilerimizin biçimini kontrol edeceğiz.

In [12]:
# ADIM 4.1 — Baseline için verileri hazırlama

import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# Orijinal metinler
X_train = ds["train"]["text"]
X_val = ds["validation"]["text"]
X_test = ds["test"]["text"]

# Duygu etiketlerini 28 sütunlu 0/1 biçimine dönüştür
mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))

y_train = mlb.fit_transform(ds["train"]["labels"])
y_val = mlb.transform(ds["validation"]["labels"])
y_test = mlb.transform(ds["test"]["labels"])

print("Eğitim metinleri:", len(X_train))
print("Doğrulama metinleri:", len(X_val))
print("Test metinleri:", len(X_test))

print("\nEğitim etiket matrisi:", y_train.shape)

print("\nÖrnek yorum:", X_train[0])
print("Gerçek duygular:", [label_names[i] for i in np.where(y_train[0] == 1)[0]])
print("Sayısal etiket:", y_train[0])

Eğitim metinleri: 43410
Doğrulama metinleri: 5426
Test metinleri: 5427

Eğitim etiket matrisi: (43410, 28)

Örnek yorum: My favourite food is anything I didn't have to cook myself.
Gerçek duygular: ['neutral']
Sayısal etiket: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]


### ADIM 4.1 — Etiket matrisi: gerçek çıktıların açıklaması

| Çıktı | Anlamı |
|---|---|
| 43.410 eğitim metni | Eğitim yorumları |
| 5.426 doğrulama metni | Eşik seçimi |
| 5.427 test metni | Nihai karşılaştırma |
| (43410, 28) | Her yorum için 28 ikili duygu etiketi |

İlk gerçek örnek `My favourite food is anything I didn't have to cook myself.` veri setinde `neutral` etiketlidir. Etiket vektöründe yalnızca son eleman 1 olur.

**Proje formu:** Kutu 3 (Data) ve Kutu 4 (Method).

## ADIM 4.2 — Baseline Modelinin Eğitilmesi

### Amaç
GoEmotions veri seti üzerinde TF-IDF + Logistic Regression yöntemini kullanarak ilk duygu sınıflandırma modelimizi eğitmek.

### Neden yapıyoruz?
Araştırmamızda Transformer tabanlı modelin yazım hatalarına karşı dayanıklılığını karşılaştırabilmek için bir başlangıç modeline (baseline) ihtiyacımız var.

### Kullanılan yöntem
- **TF-IDF:** Yorum metinlerini sayısal özelliklere dönüştürür.
- **Logistic Regression:** Metin özelliklerinden duygu etiketlerini öğrenir.
- **One-vs-Rest:** Her duygu için ayrı sınıflandırıcı kullanarak çok etiketli tahmin yapılmasını sağlar.

### Eğitim koşulları
- **Eğitim verisi:** 43.410 orijinal GoEmotions yorumu.
- **Hedef:** 28 duygu etiketinin tahmin edilmesi.
- **Test verisi:** Eğitim sırasında kullanılmaz.
- **Tekrarlanabilirlik:** Sabit `random_state` kullanılır.

### Bu aşamadan beklenen sonuç
Eğitilmiş bir baseline model ve TF-IDF özellik matrisi elde etmek.

Modelin başarısı bir sonraki aşamada Macro-F1 kullanılarak ölçülecektir.

### Proje önerisiyle ilişkisi
**Kutu 4 — Method and Baseline:** Baseline modelinin kurulması ve eğitilmesi.

In [13]:
# ADIM 4.2 — TF-IDF + Logistic Regression eğitimi

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
import time

baslangic = time.time()

# 1. Metinleri sayısal özelliklere dönüştür
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)

print("TF-IDF matrisi:", X_train_tfidf.shape)

# 2. Çok etiketli duygu sınıflandırma modelini oluştur
baseline = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    n_jobs=1
)

# 3. Modeli orijinal eğitim verisiyle eğit
print("Model eğitiliyor...")

baseline.fit(X_train_tfidf, y_train)

print("Model eğitimi tamamlandı.")
print(f"Eğitim süresi: {time.time() - baslangic:.1f} saniye")

TF-IDF matrisi: (43410, 20000)
Model eğitiliyor...
Model eğitimi tamamlandı.
Eğitim süresi: 24.9 saniye


### ADIM 4.2 — Eğitim Çıktısının Kaydı

TF-IDF + Logistic Regression baseline modeli, GoEmotions veri setindeki **43.410 eğitim yorumu** kullanılarak eğitilmiştir.

Metinler, **20.000 TF-IDF özelliği** ile sayısal olarak temsil edilmiştir.

**Güncel çalıştırma sonuçları:**
- Eğitim verisi: 43.410 yorum
- TF-IDF matris boyutu: (43410, 20000)
- Model: TF-IDF + Logistic Regression
- Eğitim süresi: **22,1 saniye**
- Eğitim durumu: Başarıyla tamamlandı.

**Değerlendirme:** Bu aşamada yalnızca model eğitimi gerçekleştirilmiştir. Modelin sınıflandırma başarısı henüz ölçülmemiştir. Macro-F1 ve Micro-F1 değerleri sonraki doğrulama ve test aşamalarında hesaplanacaktır.

**Not:** Eğitim süresi Colab çalışma ortamına göre değişebilir. Bu nedenle 22,1 saniye, mevcut çalıştırmaya ait gözlemdir.

## ADIM 4.3 — Baseline Modelinin Doğrulama Kümesinde Değerlendirilmesi

### Amaç
Eğitilen TF-IDF + Logistic Regression modelinin, eğitimde kullanılmamış doğrulama yorumlarında ne kadar başarılı olduğunu ölçmek ve uygun karar eşiğini belirlemek.

### Neden yapıyoruz?
Model her duygu için bir olasılık üretir. Bu olasılığın hangi değerden itibaren pozitif etiket sayılacağını belirlememiz gerekir.

Karar eşiğini test verisi üzerinde seçmek sonuçları yanıltabileceğinden, bu işlemi doğrulama kümesinde yapıyoruz.

### Değerlendirme metriği
**Macro-F1:** Her duygu için ayrı hesaplanan F1 puanlarının eşit ağırlıklı ortalamasıdır. Az görülen duyguların başarısını da dikkate almamızı sağlar.

### Yöntem
- 5.426 orijinal doğrulama yorumu kullanılır.
- Birden fazla karar eşiği denenir.
- En yüksek Macro-F1 puanını veren eşik seçilir.
- Seçilen eşik daha sonraki test deneylerinde sabit tutulur.

### Proje önerisiyle ilişkisi
**Kutu 5 — Evaluation:** Değerlendirme metriğinin ve karar eşiği seçim yönteminin belirlenmesi.

**Kutu 6 — First Look:** Modelden elde edilen ilk gerçek performans sonuçlarının hazırlanması.

In [14]:
# ADIM 4.3 — Doğrulama kümesi ve karar eşiği seçimi

import numpy as np
from sklearn.metrics import f1_score

# 1. Doğrulama metinlerini TF-IDF ile dönüştür
X_val_tfidf = tfidf.transform(X_val)

# 2. Her duygu için tahmin olasılıklarını hesapla
val_olasilik = baseline.predict_proba(X_val_tfidf)

print("Tahmin matrisi:", val_olasilik.shape)

# 3. Farklı karar eşiklerini karşılaştır
thresholds = [
    0.10, 0.15, 0.20, 0.25, 0.30,
    0.35, 0.40, 0.45, 0.50, 0.55,
    0.60, 0.65, 0.70, 0.75, 0.80
]

sonuclar = []

for esik in thresholds:
    tahminler = (val_olasilik >= esik).astype(int)

    macro = f1_score(
        y_val,
        tahminler,
        average="macro",
        zero_division=0
    )

    micro = f1_score(
        y_val,
        tahminler,
        average="micro",
        zero_division=0
    )

    sonuclar.append((esik, macro, micro))

    print(
        f"Eşik: {esik:.2f} | "
        f"Macro-F1: {macro:.4f} | "
        f"Micro-F1: {micro:.4f}"
    )

# 4. Macro-F1'e göre en iyi eşiği seç
en_iyi_esik, en_iyi_macro, en_iyi_micro = max(
    sonuclar,
    key=lambda x: x[1]
)

print("\nEN İYİ DOĞRULAMA SONUCU")
print("Seçilen eşik:", en_iyi_esik)
print(f"Macro-F1: {en_iyi_macro:.4f}")
print(f"Micro-F1: {en_iyi_micro:.4f}")

Tahmin matrisi: (5426, 28)
Eşik: 0.10 | Macro-F1: 0.1595 | Micro-F1: 0.1812
Eşik: 0.15 | Macro-F1: 0.2157 | Micro-F1: 0.2343
Eşik: 0.20 | Macro-F1: 0.2656 | Micro-F1: 0.2840
Eşik: 0.25 | Macro-F1: 0.3091 | Micro-F1: 0.3291
Eşik: 0.30 | Macro-F1: 0.3452 | Micro-F1: 0.3717
Eşik: 0.35 | Macro-F1: 0.3738 | Micro-F1: 0.4080
Eşik: 0.40 | Macro-F1: 0.4026 | Micro-F1: 0.4425
Eşik: 0.45 | Macro-F1: 0.4220 | Micro-F1: 0.4687
Eşik: 0.50 | Macro-F1: 0.4391 | Micro-F1: 0.4881
Eşik: 0.55 | Macro-F1: 0.4503 | Micro-F1: 0.5030
Eşik: 0.60 | Macro-F1: 0.4596 | Micro-F1: 0.5075
Eşik: 0.65 | Macro-F1: 0.4636 | Micro-F1: 0.5021
Eşik: 0.70 | Macro-F1: 0.4563 | Micro-F1: 0.4784
Eşik: 0.75 | Macro-F1: 0.4461 | Micro-F1: 0.4580
Eşik: 0.80 | Macro-F1: 0.4236 | Micro-F1: 0.4217

EN İYİ DOĞRULAMA SONUCU
Seçilen eşik: 0.65
Macro-F1: 0.4636
Micro-F1: 0.5021


## ADIM 4.3.1 — Karar Eşiği Aralığının Genişletilmesi

### Amaç
Baseline modelinin doğrulama kümesindeki Macro-F1 performansını artırabilecek karar eşiklerini araştırmak.

### Neden yapıyoruz?
İlk deneyde 0,10–0,50 arasındaki eşikler denenmiş ve en yüksek Macro-F1 değeri 0,50 eşiğinde elde edilmiştir.

Ancak performansın 0,50'ye kadar sürekli yükselmesi, daha yüksek eşiklerin de incelenmesini gerektirmiştir.

### Yöntem
- Daha önce eğitilen baseline modeli değiştirilmez.
- Aynı 5.426 doğrulama yorumu kullanılır.
- 0,50–0,85 aralığındaki ek karar eşikleri denenir.
- En yüksek Macro-F1 sonucunu veren eşik belirlenir.
- Seçilen eşik, daha sonraki orijinal ve yazım hatalı test değerlendirmelerinde sabit tutulur.

### İlk doğrulama sonuçları
- İlk denenen en iyi eşik: **0,50**
- Macro-F1: **0,4388**
- Micro-F1: **0,4875**

### Proje önerisiyle ilişkisi
**Kutu 4 — Method and Baseline:** Baseline modelinin karar mekanizmasının belirlenmesi.

**Kutu 5 — Evaluation:** Test kümesini kullanmadan, doğrulama verisi üzerinde karar eşiğinin seçilmesi.

In [15]:
# ADIM 4.3.1 — Karar eşiği aralığını genişletme

yeni_esikler = np.round(np.arange(0.50, 0.86, 0.05), 2)

ek_sonuclar = []

for esik in yeni_esikler:
    tahminler = (val_olasilik >= esik).astype(int)

    macro = f1_score(
        y_val,
        tahminler,
        average="macro",
        zero_division=0
    )

    micro = f1_score(
        y_val,
        tahminler,
        average="micro",
        zero_division=0
    )

    ek_sonuclar.append((float(esik), macro, micro))

    print(
        f"Eşik: {esik:.2f} | "
        f"Macro-F1: {macro:.4f} | "
        f"Micro-F1: {micro:.4f}"
    )

# Önceki ve yeni sonuçları birlikte değerlendir
tum_sonuclar = sonuclar + ek_sonuclar

en_iyi_esik, en_iyi_macro, en_iyi_micro = max(
    tum_sonuclar,
    key=lambda x: x[1]
)

print("\nGENİŞLETİLMİŞ ARALIKTA EN İYİ SONUÇ")
print("Seçilen eşik:", en_iyi_esik)
print(f"Macro-F1: {en_iyi_macro:.4f}")
print(f"Micro-F1: {en_iyi_micro:.4f}")

Eşik: 0.50 | Macro-F1: 0.4391 | Micro-F1: 0.4881
Eşik: 0.55 | Macro-F1: 0.4503 | Micro-F1: 0.5030
Eşik: 0.60 | Macro-F1: 0.4596 | Micro-F1: 0.5075
Eşik: 0.65 | Macro-F1: 0.4636 | Micro-F1: 0.5021
Eşik: 0.70 | Macro-F1: 0.4563 | Micro-F1: 0.4784
Eşik: 0.75 | Macro-F1: 0.4461 | Micro-F1: 0.4580
Eşik: 0.80 | Macro-F1: 0.4236 | Micro-F1: 0.4217
Eşik: 0.85 | Macro-F1: 0.3994 | Micro-F1: 0.3859

GENİŞLETİLMİŞ ARALIKTA EN İYİ SONUÇ
Seçilen eşik: 0.65
Macro-F1: 0.4636
Micro-F1: 0.5021


### ADIM 4.3.2 — Doğrulama Eşiğinin Otomatik Raporlanması

TF-IDF + Logistic Regression baseline modelinin karar eşiği, GoEmotions doğrulama kümesindeki **5.426 yorum** üzerinden belirlenmiştir.

Güncel çalıştırmada en yüksek **Macro-F1 = 0,4636**, **0,65 karar eşiğinde** elde edilmiştir. Genişletilmiş eşik aralığında yapılan kontrol de bu sonucu doğrulamıştır.

**Macro-F1**, temel değerlendirme ve eşik seçim metriğidir. **Micro-F1 = 0,5021** ise tamamlayıcı performans göstergesi olarak raporlanmıştır.

Seçilen **0,65 eşiği**, sonraki test deneylerinde sabit tutulacaktır. Test verileri eşik seçimi amacıyla kullanılmayacaktır.

**Not:** Bu değerler mevcut çalıştırmaya aittir. Notebook yeniden çalıştırıldığında güncel sonuçlar ilgili kod hücresinin çıktısından kontrol edilmelidir.

In [16]:
print(f'GÜNCEL doğrulama: eşik={en_iyi_esik:.2f}, macro-F1={en_iyi_macro:.4f}, micro-F1={en_iyi_micro:.4f}')
assert 0 < en_iyi_esik < 1

GÜNCEL doğrulama: eşik=0.65, macro-F1=0.4636, micro-F1=0.5021


## ADIM 4.4 — Baseline: orijinal ve %10 yazım hatalı test

**Neden?** Aynı 5.427 yorum, aynı gerçek etiketler, aynı model ve doğrulamada seçilen **0,65** eşiğiyle iki kez değerlendirilecek. Testte yeniden eğitim veya eşik ayarlama **yok**. **Form:** Kutu 5 ve 6. Önceki sonuçları kaydetmek yerine bu hücre gerçek test sonuçlarını üretir.

In [17]:
from sklearn.metrics import f1_score
ESIK_BASELINE = float(en_iyi_esik)
print('Doğrulama kümesinde seçilen baseline eşiği:', ESIK_BASELINE)
X_test_hatali = [yazim_hatasi_ekle(metin, oran=0.10, seed=i)[0] for i, metin in enumerate(X_test)]
assert len(X_test_hatali)==len(X_test)==len(y_test)
assert all('[NAME]' not in a or '[NAME]' in b for a,b in zip(X_test,X_test_hatali))
X_test_tfidf = tfidf.transform(X_test)
X_test_hatali_tfidf = tfidf.transform(X_test_hatali)
base_prob_orig = baseline.predict_proba(X_test_tfidf)
base_prob_typo = baseline.predict_proba(X_test_hatali_tfidf)
base_pred_orig = (base_prob_orig>=ESIK_BASELINE).astype(int)
base_pred_typo = (base_prob_typo>=ESIK_BASELINE).astype(int)
def metrics(y,p):
    return {'macro_f1':float(f1_score(y,p,average='macro',zero_division=0)), 'micro_f1':float(f1_score(y,p,average='micro',zero_division=0))}
base_original = metrics(y_test,base_pred_orig)
base_typo = metrics(y_test,base_pred_typo)
base_drop = base_original['macro_f1']-base_typo['macro_f1']
print('BASELINE ORİJİNAL:',base_original)
print('BASELINE YAZIM HATALI:',base_typo)
print(f'Macro-F1 kaybı: {base_drop:.4f} ({base_drop*100:.2f} yüzde puan)')

Doğrulama kümesinde seçilen baseline eşiği: 0.65
BASELINE ORİJİNAL: {'macro_f1': 0.44730323545887535, 'micro_f1': 0.499546553808948}
BASELINE YAZIM HATALI: {'macro_f1': 0.43201297328332483, 'micro_f1': 0.4811127156646652}
Macro-F1 kaybı: 0.0153 (1.53 yüzde puan)


### ADIM 4.4 — Gerçek Test Sonuçlarının Kaydı

TF-IDF + Logistic Regression baseline modeli, GoEmotions veri setindeki **5.427 test yorumu** üzerinde değerlendirilmiştir.

Karar eşiği, doğrulama kümesinde belirlenen **0,65** değerinde sabit tutulmuştur.

| Performans metriği | Orijinal metin | %10 yazım hatalı metin |
|---|---:|---:|
| Macro-F1 | 0,4473 | 0,4320 |
| Micro-F1 | 0,4995 | 0,4811 |

**Temel sonuç:** Sentetik yazım hataları nedeniyle Macro-F1 performansı **0,0153 (1,53 yüzde puan)** azalmıştır.

Bu sonuç, baseline modelinin yazım hatalarından etkilendiğini göstermektedir. Ancak **Transformer modelinin daha dayanıklı olduğunu kanıtlamaz**. Bunun için DistilBERT modelinin aynı test koşullarında değerlendirilmesi gerekmektedir.

**Not:** Bu değerler mevcut çalıştırmanın gerçek çıktılarıdır. Notebook yeniden çalıştırıldığında güncel sonuçlar kod hücresinden kontrol edilmelidir.

In [18]:
baseline_results={'model':'TF-IDF + OneVsRest Logistic Regression','threshold':ESIK_BASELINE,'original':base_original,'typo_10pct':base_typo,'macro_f1_drop':base_drop,'n_test':len(X_test)}
(OUTPUT_DIR/'baseline_results.json').write_text(json.dumps(baseline_results,indent=2),encoding='utf-8')
print(json.dumps(baseline_results,indent=2))

{
  "model": "TF-IDF + OneVsRest Logistic Regression",
  "threshold": 0.65,
  "original": {
    "macro_f1": 0.44730323545887535,
    "micro_f1": 0.499546553808948
  },
  "typo_10pct": {
    "macro_f1": 0.43201297328332483,
    "micro_f1": 0.4811127156646652
  },
  "macro_f1_drop": 0.015290262175550517,
  "n_test": 5427
}


## ADIM 4.5 — Üç Yazım Hatası Tekrarı ve Always-Neutral Referansı

### Deneyin Amacı

TF-IDF + Logistic Regression baseline modelinin yazım hatalarına karşı dayanıklılığını daha güvenilir değerlendirmek için **üç farklı rastgele gürültü tohumu (seed = 0, 1, 2)** kullanılacaktır.

Her tekrarda aynı test yorumlarına yaklaşık %10 oranında sentetik yazım hatası uygulanacak, ancak değiştirilen kelimeler farklılaşabilecektir.

İlk tekrar (`seed = 0`), ADIM 4.4'teki ana deneyle aynı tutulacaktır.

### Always-Neutral Referansı

Ek olarak, bütün test yorumlarını `neutral` olarak sınıflandıran basit bir referans model değerlendirilecektir.

Bu referans, TF-IDF + Logistic Regression modelinin çok basit bir tahmin stratejisine kıyasla ne kadar başarılı olduğunu anlamamızı sağlayacaktır.

### Değerlendirme

Her gürültü tohumu için yazım hatalı Macro-F1 skoru ve orijinal metinlere göre performans kaybı hesaplanacaktır.

Üç tekrarın ortalama performans kaybı ve standart sapması raporlanacaktır.

**Önemli:** Üç farklı gürültü tohumu kullanılması, bootstrap güven aralığının yerine geçmez. Bu tekrarlar aynı test kümesinde farklı yazım hatası örneklerinin etkisini ölçmektedir.

In [19]:
NOISE_SEEDS = [0, 1, 2]
noise_texts = {0: X_test_hatali}
noise_stats = []
base_noise_preds = {0: base_pred_typo}
for g in NOISE_SEEDS:
    if g != 0:
        noise_texts[g] = [yazim_hatasi_ekle(t, oran=0.10, seed=g*1_000_003+i)[0] for i,t in enumerate(X_test)]
        base_noise_preds[g] = (baseline.predict_proba(tfidf.transform(noise_texts[g])) >= ESIK_BASELINE).astype(int)
    changed, eligible, changed_comments = 0, 0, 0
    for i,t in enumerate(X_test):
        _, d, u = yazim_hatasi_ekle(t, oran=0.10, seed=g*1_000_003+i)
        changed += d; eligible += u; changed_comments += int(d > 0)
    noise_stats.append({'seed':g, 'eligible_words':eligible,'changed_words':changed,
                        'realized_rate':changed/eligible,'changed_comments':changed_comments,
                        'baseline_typo_macro_f1':metrics(y_test,base_noise_preds[g])['macro_f1'],
                        'baseline_drop':base_original['macro_f1']-metrics(y_test,base_noise_preds[g])['macro_f1']})
noise_df = pd.DataFrame(noise_stats)
noise_df.to_csv(OUTPUT_DIR/'three_noise_seeds_baseline.csv',index=False)
neutral_predictions = np.zeros_like(y_test)
neutral_predictions[:,label_names.index('neutral')] = 1
neutral_reference = metrics(y_test,neutral_predictions)
print('Always-neutral referans:',neutral_reference)
print(noise_df.to_string(index=False))
print('Ortalama baseline kaybı:',noise_df.baseline_drop.mean(), 'std:',noise_df.baseline_drop.std(ddof=1))

Always-neutral referans: {'macro_f1': 0.017693770050299023, 'micro_f1': 0.30401497107859815}
 seed  eligible_words  changed_words  realized_rate  changed_comments  baseline_typo_macro_f1  baseline_drop
    0           37661           3719       0.098749              2587                0.432013       0.015290
    1           37661           3676       0.097608              2579                0.421954       0.025349
    2           37661           3757       0.099758              2583                0.426027       0.021277
Ortalama baseline kaybı: 0.020638746179320416 std: 0.0050597529401625535


### ADIM 4.5 — Deney Sonuçları ve Değerlendirme

TF-IDF + Logistic Regression baseline modelinin yazım hatalarına karşı dayanıklılığı, GoEmotions test kümesinde üç farklı rastgele gürültü tohumu (`seed = 0, 1, 2`) kullanılarak değerlendirilmiştir.

| Seed | Değiştirilen kelime | Yazım hatası oranı | Macro-F1 | Macro-F1 kaybı |
|---|---:|---:|---:|---:|
| 0 | 3.719 | %9,87 | 0,4320 | 0,0153 |
| 1 | 3.676 | %9,76 | 0,4220 | 0,0253 |
| 2 | 3.757 | %9,98 | 0,4260 | 0,0213 |

**Orijinal test Macro-F1:** 0,4473

**Ortalama yazım hatalı Macro-F1:** 0,4267

**Ortalama Macro-F1 kaybı:** 0,0206 (2,06 yüzde puan)

**Performans kaybının standart sapması:** 0,0051

### Always-Neutral Referansı

Bütün yorumları `neutral` olarak sınıflandıran referans modelin sonuçları:

- Macro-F1: **0,0177**
- Micro-F1: **0,3040**

TF-IDF + Logistic Regression modeli, bu basit referans sınıflandırıcıdan daha yüksek performans elde etmiştir.

### Sonuçların Yorumlanması

Üç farklı yazım hatası uygulamasında da baseline modelinin Macro-F1 performansı azalmıştır. Ancak performans kaybının büyüklüğü, değiştirilen kelimelere göre farklılık göstermiştir.

Bu bulgu, dayanıklılık değerlendirmesinde tek bir rastgele yazım hatası uygulamasına bağlı kalmamanın önemini göstermektedir.

**Sınırlılık:** Üç gürültü tekrarı, istatistiksel anlamlılık testi veya bootstrap güven aralığının yerine geçmez. Ayrıca bu sonuçlar henüz DistilBERT ile karşılaştırma içermemektedir.

### ADIM 4.5 — Duygu bazında baseline analizi

**Neden?** Macro-F1 düşüşünün hangi etiketlerde yoğunlaştığını görürüz. Küçük destekli sınıfların F1 değişimleri oynak olabilir; bu sonuçlar keşifseldir.

In [20]:
from sklearn.metrics import f1_score

baseline_label_orig = f1_score(y_test, base_pred_orig, average=None, zero_division=0)
baseline_label_typo = f1_score(y_test, base_pred_typo, average=None, zero_division=0)

baseline_emotions_df = pd.DataFrame({
    'emotion': label_names,
    'support': y_test.sum(axis=0),
    'original_f1': baseline_label_orig,
    'typo_f1': baseline_label_typo,
    'drop': baseline_label_orig - baseline_label_typo
}).sort_values('drop', ascending=False)

baseline_emotions_df.to_csv(
    OUTPUT_DIR / 'baseline_per_emotion.csv',
    index=False
)

print(baseline_emotions_df.to_string(index=False))

       emotion  support  original_f1  typo_f1      drop
         grief        6     0.461538 0.400000  0.061538
          love      238     0.759295 0.707819  0.051477
     gratitude      352     0.895480 0.848214  0.047266
         anger      198     0.415704 0.375309  0.040396
    admiration      504     0.630561 0.592000  0.038561
       remorse       56     0.652778 0.618705  0.034073
     confusion      153     0.369830 0.341346  0.028484
      optimism      186     0.537530 0.509901  0.027629
           joy      161     0.546419 0.522667  0.023752
       sadness      156     0.481586 0.458924  0.022663
     amusement      264     0.800000 0.779599  0.020401
          fear       78     0.637500 0.617284  0.020216
    excitement      103     0.333333 0.314685  0.018648
       disgust      123     0.475096 0.460938  0.014158
     annoyance      320     0.297297 0.283159  0.014138
      approval      351     0.310300 0.297927  0.012372
        caring      135     0.254335 0.244048  0

### ADIM 4.5 — Duygu Bazında Baseline Sonuçları

TF-IDF + Logistic Regression modelinin performansı, GoEmotions veri setindeki 28 duygu kategorisi için ayrı ayrı değerlendirilmiştir.

**En yüksek F1 düşüşleri:**

| Duygu | Test desteği | Orijinal F1 | Yazım hatalı F1 | F1 kaybı |
|---|---:|---:|---:|---:|
| Grief | 6 | 0,4615 | 0,4000 | 0,0615 |
| Love | 238 | 0,7593 | 0,7078 | 0,0515 |
| Gratitude | 352 | 0,8955 | 0,8482 | 0,0473 |
| Anger | 198 | 0,4157 | 0,3753 | 0,0404 |
| Admiration | 504 | 0,6306 | 0,5920 | 0,0386 |

### Bulguların Yorumlanması

Yazım hatalarının etkisi duygu kategorileri arasında farklılık göstermektedir. Özellikle `love`, `gratitude`, `anger` ve `admiration` kategorilerinde belirgin F1 düşüşleri gözlemlenmiştir.

`grief` kategorisinde en yüksek düşüş görülmesine rağmen yalnızca **6 test örneği** bulunduğundan bu sonuç dikkatli yorumlanmalıdır.

Bazı kategorilerde F1 skorunun hafifçe artması, sentetik yazım hatalarının belirli tahminleri değiştirmesiyle ilişkili olabilir. Ancak genel Macro-F1 performansı azalmıştır.

**Sınırlılık:** Bu analiz tek bir yazım hatası tohumu (`seed = 0`) üzerinden yapılmıştır. Duygu bazındaki farklılıklar keşifsel bulgulardır; istatistiksel olarak anlamlı farklılıklar olarak değerlendirilmemelidir.

### Araştırmaya Katkısı

Bu analiz, yazım hatalarının tüm duygu kategorilerini aynı ölçüde etkilemediğini göstermektedir. DistilBERT değerlendirmesi tamamlandığında iki modelin duygu bazındaki performans değişimleri de karşılaştırılacaktır.

In [21]:
import sympy
print("SymPy sürümü:", sympy.__version__)
print("SymPy dosyası:", sympy.__file__)

import sympy.printing
print("SymPy printing modülü başarıyla yüklendi.")

SymPy sürümü: 1.14.0
SymPy dosyası: /usr/local/lib/python3.13/dist-packages/sympy/__init__.py
SymPy printing modülü başarıyla yüklendi.


In [22]:
import sympy
import sympy.printing

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

print("PyTorch:", torch.__version__)
print("SymPy:", sympy.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Transformers başarıyla yüklendi!")

PyTorch: 2.14.1+cu130
SymPy: 1.14.0
GPU: Tesla T4
Transformers başarıyla yüklendi!


## ADIM 4.6 — Yazım hatası üreticisi testleri ve veri kontrolleri (Tarık'ın görevi)

**Neden?** Deney sonuçlarına güvenebilmek için yazım hatası üreticisinin doğru çalıştığını ve train/dev/test arasında sızıntı olmadığını kontrol ederiz. Bu hücre deney kodunu değiştirmez.

In [ ]:
# ADIM 4.6 — Yazım hatası üreticisi testleri + veri sızıntısı ve etiket dağılımı kontrolü (Tarık'ın görevi)
# Deney kodunu DEĞİŞTİRMEZ; sadece kontrol eder. Bir kontrol başarısız olursa hata verir.
import re
from collections import Counter
import numpy as np

PARCA = r"\[[^\[\]\n]*\]|[A-Za-z]+|[^A-Za-z\[\]]+|[\[\]]"
test_metinleri = list(ds["test"]["text"])

# --- 1) Yazım hatası üreticisi testleri
# Test 1: oran 0 ise hiçbir şey değişmez
assert all(yazim_hatasi_ekle(t, oran=0.0, seed=i)[0] == t for i, t in enumerate(test_metinleri[:500]))
# Test 2: aynı seed her zaman aynı sonucu verir (tekrar üretilebilirlik)
assert all(yazim_hatasi_ekle(t, seed=i) == yazim_hatasi_ekle(t, seed=i) for i, t in enumerate(test_metinleri[:500]))
# Test 3: [NAME], [RELIGION] gibi işaretler oran 1.0'da bile korunur
ornek = "I really appreciate [NAME] and respect [RELIGION]. This is wonderful!"
cikti = yazim_hatasi_ekle(ornek, oran=1.0, seed=42)[0]
assert "[NAME]" in cikti and "[RELIGION]" in cikti
# Test 4: değişen her kelime 4+ harfli, ilk harfi aynı, uzunluğu en fazla 1 farklı
kontrol_edilen = son_harf_degisen = 0
for i, t in enumerate(test_metinleri):
    a = re.findall(PARCA, t)
    b = re.findall(PARCA, yazim_hatasi_ekle(t, oran=0.10, seed=i)[0])
    assert len(a) == len(b), i
    for x, y in zip(a, b):
        if x != y:
            assert len(x) >= 4 and x[0] == y[0] and abs(len(x) - len(y)) <= 1, (i, x, y)
            kontrol_edilen += 1
            son_harf_degisen += x[-1] != y[-1]
print(f"Yazım hatası üreticisi: 4 test geçti ({kontrol_edilen:,} değişen kelime kontrol edildi; "
      f"{son_harf_degisen} tanesinde sondaki yer değiştirme son harfi oynattı)")

# --- 2) Veri sızıntısı: bölümler arasında ortak id veya aynı metin var mı?
for a_ad, b_ad in [("train", "test"), ("train", "validation"), ("validation", "test")]:
    ortak_id = set(ds[a_ad]["id"]) & set(ds[b_ad]["id"])
    ortak_metin = set(ds[a_ad]["text"]) & set(ds[b_ad]["text"])
    assert not ortak_id, f"{a_ad}/{b_ad} ortak id var!"
    print(f"{a_ad}/{b_ad}: ortak id 0 | birebir aynı metin {len(ortak_metin)}")

# --- 3) Etiket dağılımı: her duygunun yorumlardaki payı, train ile test arasında
def dagilim(bolum):
    sayac = Counter(k for etiketler in ds[bolum]["labels"] for k in etiketler)
    return np.array([sayac[k] / len(ds[bolum]["labels"]) for k in range(len(label_names))])
d_train, d_test = dagilim("train"), dagilim("test")
en_kotu = int(np.argmax(np.abs(d_train - d_test)))
print(f"Etiket dağılımı: train ile test arasındaki en büyük fark {100 * abs(d_train - d_test).max():.2f} puan ({label_names[en_kotu]})")
print("TÜM KONTROLLER GEÇTİ")


## ADIM 4.7 — Baseline hata analizi

**Neden?** Yazım hatası baseline'ı nerede yanıltıyor? Slayt 5 ve Project Card alan 9'daki sayılar bu hücreden gelir.

In [ ]:
# ADIM 4.7 — Baseline hata analizi (slayt 5 ve Project Card alan 9 bu hücreden gelir)
# A) Yazım hatasının neden olduğu değişiklikler (yazım hatası çekilişi 0)
# B) Temiz testteki hatalar (yazım hatası olmadan)
import re
import numpy as np
import pandas as pd

GRUP = {  # duyguların kabaca kutbu: neg / pos / amb
    "anger": "neg", "annoyance": "neg", "disapproval": "neg", "disgust": "neg", "disappointment": "neg",
    "sadness": "neg", "grief": "neg", "remorse": "neg", "embarrassment": "neg", "fear": "neg", "nervousness": "neg",
    "joy": "pos", "amusement": "pos", "excitement": "pos", "love": "pos", "admiration": "pos", "approval": "pos",
    "caring": "pos", "gratitude": "pos", "optimism": "pos", "pride": "pos", "relief": "pos", "desire": "pos",
    "surprise": "amb", "realization": "amb", "confusion": "amb", "curiosity": "amb"}

# Her duygu için logistic regression ağırlığı en büyük 30 tek kelime = o duygunun "anahtar kelimeleri"
sozluk = np.array(tfidf.get_feature_names_out())
tek_kelime = np.array([" " not in v for v in sozluk])
anahtar = {}
for k, est in enumerate(baseline.estimators_):
    w = est.coef_[0].copy(); w[~tek_kelime] = -np.inf
    anahtar[label_names[k]] = set(sozluk[np.argsort(w)[::-1][:30]])

# --- A) Yazım hatası kaynaklı değişiklikler
satirlar_a = []
for i in range(len(X_test)):
    gold = set(np.where(y_test[i])[0]); c = set(np.where(base_pred_orig[i])[0]); t = set(np.where(base_pred_typo[i])[0])
    bozulan = [(a, b) for a, b in zip(re.findall(r"[A-Za-z]+", X_test[i]), re.findall(r"[A-Za-z]+", X_test_hatali[i])) if a != b]
    for k in (gold & c) - t:
        kw = [a for a, _ in bozulan if a.lower() in anahtar[label_names[k]]]
        satirlar_a.append({"test_index": i, "emotion": label_names[k], "keyword": ", ".join(kw), "text_typo": X_test_hatali[i],
                           "category": "A1 anahtar kelime bozuldu, doğru etiket kayboldu" if kw else "A2 anahtar kelime bozulmadan doğru etiket kayboldu"})
    for k in (t - c) - gold:
        satirlar_a.append({"test_index": i, "emotion": label_names[k], "keyword": "", "text_typo": X_test_hatali[i],
                           "category": "A3 yazım hatasından sonra yeni yanlış etiket"})
    for k in (gold & t) - c:
        satirlar_a.append({"test_index": i, "emotion": label_names[k], "keyword": "", "text_typo": X_test_hatali[i],
                           "category": "A4 yazım hatası kaçırılan etiketi tesadüfen buldu"})
hata_a = pd.DataFrame(satirlar_a)
kayip = hata_a[hata_a.category.str.startswith(("A1", "A2"))]
print(f"A) YAZIM HATASI KAYNAKLI: {len(hata_a)} etiket değişikliği, {hata_a.test_index.nunique()} yorumda")
print(hata_a.category.value_counts().sort_index().to_string())
print(f"Kaybolan doğru etiketlerin %{100 * (kayip.category.str.startswith('A1')).mean():.0f}'sında anahtar kelime bozulmuş "
      f"({(kayip.category.str.startswith('A1')).sum()} / {len(kayip)})")
print("En çok kaybolan duygular:", kayip.emotion.value_counts().head(5).to_dict())
print("\nÖrnekler:")
for kat in sorted(hata_a.category.unique()):
    r = hata_a[hata_a.category == kat].iloc[0]
    print(f"  {kat[:2]} ({r.emotion}): {r.text_typo[:90]}")

# --- B) Temiz testteki hatalar
notr = label_names.index("neutral")
satirlar_b = []
for i in range(len(X_test)):
    gold = set(np.where(y_test[i])[0]); p = set(np.where(base_pred_orig[i])[0])
    if gold == p:
        continue
    g_duygu, p_duygu = gold - {notr}, p - {notr}
    if not p: kat = "B1 hiç etiket tahmin edilmedi"
    elif g_duygu and not p_duygu: kat = "B2 duygu kaçırıldı, sadece neutral tahmin edildi"
    elif not g_duygu and p_duygu: kat = "B3 doğrusu neutral, duygu tahmin edildi"
    elif gold & p: kat = "B4 kısmen doğru (bir etiket doğru, biri eksik/fazla)"
    elif {GRUP.get(label_names[k]) for k in g_duygu} & {GRUP.get(label_names[k]) for k in p_duygu}: kat = "B5 yanlış duygu, aynı kutup"
    else: kat = "B6 yanlış duygu, ters kutup"
    satirlar_b.append({"test_index": i, "category": kat, "gold": ";".join(label_names[k] for k in sorted(gold)),
                       "pred": ";".join(label_names[k] for k in sorted(p)), "text": X_test[i]})
hata_b = pd.DataFrame(satirlar_b)
print(f"\nB) TEMİZ TESTTE HATALI YORUM: {len(hata_b)} / {len(X_test)}")
print(hata_b.category.value_counts().sort_index().to_string())


## ADIM 4.8 — Spell-check pilotu (Rümeysa'nın görevi)

**Neden?** Basit bir çözüm olarak, baseline'dan önce yazım düzeltici kullanmak kaybı geri kazandırır mı? Tek çekilişte küçük bir deneme; birkaç dakika sürer.

In [ ]:
# ADIM 4.8 — Spell-check pilotu (Rümeysa'nın görevi): baseline'dan önce yazım düzeltici işe yarar mı?
# pyspellchecker ile her kelime sözlükte yoksa en olası kelimeyle değiştirilir; [NAME] gibi işaretlere dokunulmaz.
# Tek bir yazım hatası çekilişi (0) üzerinde küçük bir deneme; ana deneyin parçası değildir.
%pip -q install pyspellchecker
import re
from spellchecker import SpellChecker

duzeltici = SpellChecker()
_onbellek = {}

def kelime_duzelt(k):
    kl = k.lower()
    if kl not in _onbellek:
        _onbellek[kl] = kl if kl in duzeltici else (duzeltici.correction(kl) or kl)
    d = _onbellek[kl]
    if k.isupper() and len(k) > 1: return d.upper()
    if k[0].isupper(): return d[0].upper() + d[1:]
    return d

def metin_duzelt(metin):
    parcalar = re.findall(r"\[[^\[\]\n]*\]|[A-Za-z]+|[^A-Za-z\[\]]+|[\[\]]", metin)
    return "".join(kelime_duzelt(p) if re.fullmatch(r"[A-Za-z]+", p) else p for p in parcalar)

def baseline_macro(metinler):
    p = (baseline.predict_proba(tfidf.transform(metinler)) >= ESIK_BASELINE).astype(int)
    return metrics(y_test, p)["macro_f1"]

temiz_duzeltilmis = [metin_duzelt(t) for t in X_test]
hatali_duzeltilmis = [metin_duzelt(t) for t in X_test_hatali]
spell_sonuc = pd.DataFrame([
    ("Temiz", base_original["macro_f1"]),
    ("Yazım hatalı", baseline_macro(X_test_hatali)),
    ("Yazım hatalı + düzeltme", baseline_macro(hatali_duzeltilmis)),
    ("Temiz + düzeltme", baseline_macro(temiz_duzeltilmis))], columns=["durum", "macro_f1"])
spell_sonuc["temize_gore_puan"] = (spell_sonuc.macro_f1 - base_original["macro_f1"]) * 100
print(spell_sonuc.round(4).to_string(index=False))
ornekler = [(a, b) for a, b in zip(X_test_hatali, hatali_duzeltilmis) if a != b][:3]
for a, b in ornekler:
    print("\nHatalı   :", a[:100]); print("Düzeltilmiş:", b[:100])


## ADIM 5 — Transformer modelini sıfırdan değil, önceden eğitilmiş DistilBERT üzerinden fine-tune etme

**Neden?** TF-IDF kelime/ngram özellikleriyle çalışırken DistilBERT bağlamsal alt-kelime temsilleri kullanır. Aynı 43.410 eğitim yorumunda 28 etiketli **çok etiketli** sınıflandırma öğrenir. Eğitim etiketleri `float32` ve kayıp `BCEWithLogitsLoss` olmalıdır. **Form:** Kutu 4, 8.

**Adil karşılaştırma:** Her iki model aynı orijinal eğitim bölümü, aynı doğrulama ve test bölümleri, aynı %10 hata üreticisi ve aynı Macro-F1 metriğiyle ölçülür. Mimari/kapasite farkları deneyin karşılaştırdığı unsurdur. **GPU:** T4 önerilir; 2 epoch yaklaşık GPU ve yük durumuna bağlıdır, kesin süre iddia edilmez.

**Önemli:** Eğitim bir süre sürebilir. `TRAIN_EPOCHS=2` ana deney; GPU yoksa eğitim çalışır ama çok yavaş olabilir. Eğitim verisini küçültmek nihai karşılaştırmanın kapsamını değiştirir; burada varsayılan olarak küçültmüyoruz.

In [23]:
%pip uninstall -y torchvision

Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130


In [24]:
from transformers.models.distilbert.modeling_distilbert import DistilBertForSequenceClassification

print("DistilBERT başarıyla içe aktarıldı!")

DistilBERT başarıyla içe aktarıldı!


In [25]:
import sys
import importlib.util

print("torchvision kurulu mu?:", importlib.util.find_spec("torchvision"))
print("Transformers bellekte mi?:", "transformers" in sys.modules)

torchvision kurulu mu?: ModuleSpec(name='torchvision', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f50f558b530>, origin='/usr/local/lib/python3.13/dist-packages/torchvision/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/torchvision'])
Transformers bellekte mi?: True


In [26]:
import json

baseline_backup = {
    "validation_threshold": float(en_iyi_esik),
    "validation_macro_f1": float(en_iyi_macro),
    "original_test": base_original,
    "typo_test": base_typo,
    "three_seed_results": noise_df.to_dict(orient="records")
}

with open("/content/baseline_backup.json", "w") as f:
    json.dump(baseline_backup, f, indent=2)

print("Baseline sonuçları yedeklendi.")

Baseline sonuçları yedeklendi.


In [27]:
import torch
import sympy
import importlib.util

print("PyTorch:", torch.__version__)
print("SymPy:", sympy.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("torchvision:", importlib.util.find_spec("torchvision"))

PyTorch: 2.14.1+cu130
SymPy: 1.14.0
GPU: Tesla T4
torchvision: ModuleSpec(name='torchvision', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f50f558b530>, origin='/usr/local/lib/python3.13/dist-packages/torchvision/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/torchvision'])


In [28]:
from transformers import AutoModelForSequenceClassification

model_test = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=28,
    problem_type="multi_label_classification"
)

print("DistilBERT başarıyla yüklendi!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT başarıyla yüklendi!


In [29]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
MODEL_ID='distilbert-base-uncased'
MAX_LENGTH=128
TRAIN_EPOCHS=2
TRAIN_BATCH_SIZE=16
EVAL_BATCH_SIZE=64
GRAD_ACCUM=2
LEARNING_RATE=3e-5
TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID)
class EmotionDataset(Dataset):
    def __init__(self,texts,labels=None):
        self.enc=TOKENIZER(list(texts),truncation=True,padding='max_length',max_length=MAX_LENGTH)
        self.labels=labels
    def __len__(self):return len(self.enc['input_ids'])
    def __getitem__(self,i):
        d={'input_ids':torch.tensor(self.enc['input_ids'][i],dtype=torch.long),
           'attention_mask':torch.tensor(self.enc['attention_mask'][i],dtype=torch.long)}
        if self.labels is not None:d['labels']=torch.tensor(self.labels[i],dtype=torch.float32)
        return d
train_ds=EmotionDataset(X_train,y_train)
val_ds=EmotionDataset(X_val,y_val)
train_loader=DataLoader(train_ds,batch_size=TRAIN_BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False)
model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,num_labels=len(label_names),problem_type='multi_label_classification',
    id2label={i:n for i,n in enumerate(label_names)},label2id={n:i for i,n in enumerate(label_names)})
model.to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=0.01)
steps_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM)
total_steps=steps_per_epoch*TRAIN_EPOCHS
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=max(1,int(0.1*total_steps)),num_training_steps=total_steps)
print('Model:',MODEL_ID,'| train batches:',len(train_loader),'| optimizer steps:',total_steps)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert-base-uncased | train batches: 2714 | optimizer steps: 2714


## ADIM 5.1 — Fine-tuning ve doğrulama kontrolü

**Neden?** Modelin parametrelerini eğitim verisinden öğreniyoruz. Her epoch sonunda doğrulama kaybını izleyip en düşük kayıplı ağırlıkları saklıyoruz. Test verisi eğitime **girmez**. Eğitimde CUDA varsa karma hassasiyet kullanılır. **Form:** Kutu 4 ve 8.

In [ ]:
import copy
@torch.no_grad()
def predict_transformer(model,loader):
    model.eval(); all_probs=[]
    for batch in tqdm(loader,desc='Transformer tahmin',leave=False):
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        logits=model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask']).logits
        all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.vstack(all_probs)
@torch.no_grad()
def val_loss_transformer(model,loader):
    model.eval(); total=0.; count=0
    for batch in loader:
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        loss=model(**batch).loss
        total+=float(loss.item())*len(batch['input_ids']);count+=len(batch['input_ids'])
    return total/count
history=[]; best_loss=float('inf'); best_state=None
scaler=torch.amp.GradScaler('cuda',enabled=(DEVICE.type=='cuda'))
train_start=time.time()
for epoch in range(TRAIN_EPOCHS):
    model.train(); optimizer.zero_grad(set_to_none=True); running=0.
    bar=tqdm(train_loader,desc=f'Epoch {epoch+1}/{TRAIN_EPOCHS}')
    for step,batch in enumerate(bar):
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        with torch.autocast(device_type=DEVICE.type,enabled=(DEVICE.type=='cuda')):
            loss=model(**batch).loss
        running+=float(loss.detach().item())
        scaler.scale(loss/GRAD_ACCUM).backward()
        if (step+1)%GRAD_ACCUM==0 or step+1==len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer);scaler.update();scheduler.step();optimizer.zero_grad(set_to_none=True)
        if step%100==0:bar.set_postfix(loss=round(running/(step+1),4))
    vl=val_loss_transformer(model,val_loader)
    history.append({'epoch':epoch+1,'train_loss':running/len(train_loader),'val_loss':vl})
    print(f'Epoch {epoch+1}: train loss={running/len(train_loader):.4f}, val loss={vl:.4f}')
    if vl<best_loss:
        best_loss=vl;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best_state);model.to(DEVICE)
training_seconds=time.time()-train_start
print('Transformer eğitim süresi (sn):',round(training_seconds,1))
pd.DataFrame(history).to_csv(OUTPUT_DIR/'transformer_training_history.csv',index=False)
model.save_pretrained(str(OUTPUT_DIR/'distilbert_goemotions_model'),safe_serialization=True)
TOKENIZER.save_pretrained(str(OUTPUT_DIR/'distilbert_goemotions_model'))
# pd.DataFrame(history).to_csv(OUTPUT_DIR/'transformer_training_history.csv',index=False)
# model.save_pretrained(str(OUTPUT_DIR/'distilbert_goemotions_model'),safe_serialization=True)
# TOKENIZER.save_pretrained(str(OUTPUT_DIR/'distilbert_goemotions_model'))

Epoch 1/2:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1: train loss=0.0804, val loss=0.0860


Epoch 2/2:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 2: train loss=0.0803, val loss=0.0860
Transformer eğitim süresi (sn): 360.3


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/goemotions_outputs/distilbert_goemotions_model/tokenizer_config.json',
 '/content/goemotions_outputs/distilbert_goemotions_model/tokenizer.json')

In [ ]:
print(torch.initial_seed())

1


## ADIM 5.2 — Transformer karar eşiğini yalnızca doğrulamada seç

**Neden?** Çok etiketli model her duygu için 0–1 arası olasılık verir. Baseline'da olduğu gibi **tek ortak eşik**, önceden belirlenmiş 0.10–0.90 ızgarasında doğrulama Macro-F1'e göre seçilir. Transformer eşiğinin baseline eşiğiyle aynı olması gerekmez; her model kendi doğrulama kümesinde ayarlanır. **Testte ayarlama yok.** **Form:** Kutu 5.

In [ ]:
transformer_val_prob=predict_transformer(model,val_loader)
THRESHOLD_GRID=np.round(np.arange(0.10,0.901,0.05),2)
transformer_val_table=[]
for t in THRESHOLD_GRID:
    pred=(transformer_val_prob>=t).astype(int)
    transformer_val_table.append({'threshold':float(t),**metrics(y_val,pred)})
transformer_val_df=pd.DataFrame(transformer_val_table)
transformer_val_df.to_csv(OUTPUT_DIR/'transformer_validation_thresholds.csv',index=False)
best_row=transformer_val_df.loc[transformer_val_df.macro_f1.idxmax()]
ESIK_TRANSFORMER=float(best_row['threshold'])
print(transformer_val_df.to_string(index=False))
print('Transformer seçilen eşik:',ESIK_TRANSFORMER,'| Macro-F1:',round(float(best_row['macro_f1']),4))

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

 threshold  macro_f1  micro_f1
      0.10  0.425992  0.538449
      0.15  0.448432  0.577393
      0.20  0.451080  0.592987
      0.25  0.445023  0.600806
      0.30  0.435696  0.599539
      0.35  0.420294  0.594322
      0.40  0.399032  0.585073
      0.45  0.374461  0.570131
      0.50  0.351389  0.554480
      0.55  0.330416  0.537949
      0.60  0.290358  0.507605
      0.65  0.248265  0.474868
      0.70  0.195556  0.435154
      0.75  0.156775  0.389210
      0.80  0.131437  0.344706
      0.85  0.109812  0.286879
      0.90  0.077026  0.190034
Transformer seçilen eşik: 0.2 | Macro-F1: 0.4511


## ADIM 6 — Transformer'ın orijinal ve yazım hatalı test performansı

**Neden?** Aynı test yorumlarının iki sürümünü aynı fine-tune edilmiş modele verip Macro-F1 farkını hesaplıyoruz. **Form:** Kutu 5, 6.

In [ ]:
test_ds_orig=EmotionDataset(X_test)
test_ds_typo=EmotionDataset(X_test_hatali)
transformer_prob_orig=predict_transformer(model,DataLoader(test_ds_orig,batch_size=EVAL_BATCH_SIZE))
transformer_prob_typo=predict_transformer(model,DataLoader(test_ds_typo,batch_size=EVAL_BATCH_SIZE))
trans_pred_orig=(transformer_prob_orig>=ESIK_TRANSFORMER).astype(int)
trans_pred_typo=(transformer_prob_typo>=ESIK_TRANSFORMER).astype(int)
trans_original=metrics(y_test,trans_pred_orig)
trans_typo=metrics(y_test,trans_pred_typo)
trans_drop=trans_original['macro_f1']-trans_typo['macro_f1']
print('TRANSFORMER ORİJİNAL:',trans_original)
print('TRANSFORMER YAZIM HATALI:',trans_typo)
print('Transformer Macro-F1 kaybı:',round(trans_drop,4))
print('Baseline Macro-F1 kaybı:',round(base_drop,4))
print('Avantaj (baseline kaybı − Transformer kaybı):',round(base_drop-trans_drop,4))

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

TRANSFORMER ORİJİNAL: {'macro_f1': 0.4419127768764328, 'micro_f1': 0.5928325292901447}
TRANSFORMER YAZIM HATALI: {'macro_f1': 0.42693841014021866, 'micro_f1': 0.5755175767680978}
Transformer Macro-F1 kaybı: 0.015
Baseline Macro-F1 kaybı: 0.0153
Avantaj (baseline kaybı − Transformer kaybı): 0.0003


In [ ]:
p_o=predict_transformer(model,DataLoader(test_ds_orig,batch_size=EVAL_BATCH_SIZE))
p_t=predict_transformer(model,DataLoader(test_ds_typo,batch_size=EVAL_BATCH_SIZE))
mo=metrics(y_test,(p_o>=0.20).astype(int))['macro_f1']
mt=metrics(y_test,(p_t>=0.20).astype(int))['macro_f1']
print('seed 1 | orig', round(mo,4), '| typo', round(mt,4), '| drop', round(mo-mt,4), '| advantage', round(base_drop-(mo-mt),4))

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

seed 1 | orig 0.4419 | typo 0.4269 | drop 0.015 | advantage 0.0003


## ADIM 6.1 — Transformer için aynı üç gürültü tekrarı

**Neden?** Baseline ile aynı bozulmuş metinleri ve aynı rastgelelik tohumlarını kullanarak üç tekrardaki dayanıklılık farkını raporluyoruz. Ana hipotez testi yine önceden belirlenen seed 0 üzerinde eşleştirilmiş bootstrap ile yapılır; diğerleri duyarlılık kontrolüdür.

In [ ]:
transformer_noise_preds = {0: trans_pred_typo}
replicate_rows = []
for g in NOISE_SEEDS:
    if g != 0:
        p = predict_transformer(model,DataLoader(EmotionDataset(noise_texts[g]),batch_size=EVAL_BATCH_SIZE))
        transformer_noise_preds[g] = (p >= ESIK_TRANSFORMER).astype(int)
    bdrop = base_original['macro_f1'] - metrics(y_test,base_noise_preds[g])['macro_f1']
    tdrop = trans_original['macro_f1'] - metrics(y_test,transformer_noise_preds[g])['macro_f1']
    replicate_rows.append({'seed':g,'baseline_drop':bdrop,'transformer_drop':tdrop,'advantage':bdrop-tdrop})
replicate_df = pd.DataFrame(replicate_rows)
replicate_df.to_csv(OUTPUT_DIR/'three_noise_seeds_model_comparison.csv',index=False)
print(replicate_df.to_string(index=False))

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

 seed  baseline_drop  transformer_drop  advantage
    0       0.015290          0.014974   0.000316
    1       0.025349          0.014505   0.010844
    2       0.021277          0.014984   0.006293


## ADIM 7 — Eşleştirilmiş bootstrap: fark gerçekten anlamlı mı?

**Neden?** Sadece iki puan arasındaki farkı görmek yetmez. Test kümesindeki yorumları **aynı indekslerle** yeniden örnekleyerek dört tahmin matrisini birlikte değerlendiriyoruz. Böylece hem iki model hem de orijinal/yazım hatalı koşullar aynı örnekler üzerinde karşılaştırılır. %95 bootstrap güven aralığı sıfırı kapsıyorsa **Transformer daha dayanıklı** sonucunu çıkarmıyoruz. **Form:** Kutu 1 ve 5.

**Fark tanımı:** `(baseline_orijinal − baseline_yazım_hatalı) − (Transformer_orijinal − Transformer_yazım_hatalı)`. Pozitif fark Transformer lehine. Bu güven aralığı bir belirsizlik ölçüsüdür; test setini tekrar tekrar ayarlama için kullanmayız.

**İstatistik notu:** Aynı bootstrap örneklem indeksleri hem orijinal hem hatalı test ve iki model için kullanılır; CI tek modelin kaybı için değil **iki kaybın farkı** içindir. %95 GA sıfırı kapsarsa üstünlük gösterilmiş sayılmaz.

In [ ]:
BOOTSTRAP_N=1000
rng=np.random.default_rng(SEED)
n=len(y_test)
bootstrap_advantages=np.empty(BOOTSTRAP_N)
for b in tqdm(range(BOOTSTRAP_N),desc='Eşleştirilmiş bootstrap'):
    ix=rng.integers(0,n,size=n)
    yy=y_test[ix]
    d_base=f1_score(yy,base_pred_orig[ix],average='macro',zero_division=0)-f1_score(yy,base_pred_typo[ix],average='macro',zero_division=0)
    d_trans=f1_score(yy,trans_pred_orig[ix],average='macro',zero_division=0)-f1_score(yy,trans_pred_typo[ix],average='macro',zero_division=0)
    bootstrap_advantages[b]=d_base-d_trans
ci_low,ci_high=np.percentile(bootstrap_advantages,[2.5,97.5])
observed_advantage=base_drop-trans_drop
supported=(observed_advantage>0 and ci_low>0)
print(f'Gözlenen avantaj: {observed_advantage:.4f}')
print(f'%95 bootstrap GA: [{ci_low:.4f}, {ci_high:.4f}]')
print('Transformer daha dayanıklı sonucunu destekliyor mu?', 'EVET' if supported else 'HAYIR / KANIT YETERSİZ')

Eşleştirilmiş bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

Gözlenen avantaj: 0.0003
%95 bootstrap GA: [-0.0088, 0.0094]
Transformer daha dayanıklı sonucunu destekliyor mu? HAYIR / KANIT YETERSİZ


## ADIM 8 — Ablasyon: yazım hatası şiddeti değişince ne oluyor?

**Neden?** Yüksek lisans araştırma liderinden istenen ek deney için %5, %10 ve %20 yazım hatası olasılığında her iki modelin Macro-F1'ini karşılaştırıyoruz. Ana hipotez **önceden belirlenmiş %10** koşuluyla test edilir; diğer oranlar destekleyici analizdir. **Form:** Kutu 7.

In [ ]:
ablation_rows=[]
for ratio in [0.05,0.10,0.20]:
    altered=[yazim_hatasi_ekle(txt,oran=ratio,seed=i)[0] for i,txt in enumerate(X_test)]
    if ratio==0.10:
        bp=base_pred_typo;tp=trans_pred_typo
    else:
        bp=(baseline.predict_proba(tfidf.transform(altered))>=ESIK_BASELINE).astype(int)
        tp=(predict_transformer(model,DataLoader(EmotionDataset(altered),batch_size=EVAL_BATCH_SIZE))>=ESIK_TRANSFORMER).astype(int)
    bm=metrics(y_test,bp)['macro_f1'];tm=metrics(y_test,tp)['macro_f1']
    ablation_rows.append({'typo_probability':ratio,'baseline_macro_f1':bm,'transformer_macro_f1':tm,'baseline_drop':base_original['macro_f1']-bm,'transformer_drop':trans_original['macro_f1']-tm})
ablation_df=pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False))
ablation_df.to_csv(OUTPUT_DIR/'ablation_typo_rates.csv',index=False)

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

Transformer tahmin:   0%|          | 0/85 [00:00<?, ?it/s]

 typo_probability  baseline_macro_f1  transformer_macro_f1  baseline_drop  transformer_drop
             0.05           0.438576              0.433702       0.008727          0.008210
             0.10           0.432013              0.426938       0.015290          0.014974
             0.20           0.405029              0.409060       0.042274          0.032853


## ADIM 9 — Duygu bazlı hata analizi ve örnek demo

**Neden?** Makro-F1 tüm duyguları eşit ağırlıklandırır; bazı duygular yazım hatalarından daha fazla etkilenebilir. Her duygu için F1 değişimini ve tahmini değişen gerçek yorumları gösteriyoruz. Kullanıcı metni üzerinde küçük demo da sunuyoruz. **Form:** Kutu 5, 6 ve final demo beklentisi.

**Etik sınır:** Reddit yorumlarında gerçek kişilerden söz edilebilir; örnekleri kısa tutuyoruz, anonimleştirilmiş özel işaretleri koruyoruz. Yapay hata bazen anlamı değiştirebilir.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
per_label=[]
for i,label in enumerate(label_names):
    row={'emotion':label,'support':int(y_test[:,i].sum())}
    for prefix,pr in [('base_orig',base_pred_orig),('base_typo',base_pred_typo),('trans_orig',trans_pred_orig),('trans_typo',trans_pred_typo)]:
        row[prefix+'_f1']=float(f1_score(y_test[:,i],pr[:,i],zero_division=0))
    row['baseline_drop']=row['base_orig_f1']-row['base_typo_f1']
    row['transformer_drop']=row['trans_orig_f1']-row['trans_typo_f1']
    per_label.append(row)
per_label_df=pd.DataFrame(per_label).sort_values('transformer_drop',ascending=False)
per_label_df.to_csv(OUTPUT_DIR/'per_emotion_analysis.csv',index=False)
print(per_label_df[['emotion','support','baseline_drop','transformer_drop']].head(12).to_string(index=False))
changed_ix=np.where((trans_pred_orig!=trans_pred_typo).any(axis=1))[0]
examples=[]
for i in changed_ix[:30]:
    examples.append({'index':int(i),'original':X_test[i],'with_typos':X_test_hatali[i],
                     'gold':', '.join(label_names[j] for j in np.where(y_test[i])[0]),
                     'pred_original':', '.join(label_names[j] for j in np.where(trans_pred_orig[i])[0]),
                     'pred_typos':', '.join(label_names[j] for j in np.where(trans_pred_typo[i])[0])})
pd.DataFrame(examples).to_csv(OUTPUT_DIR/'changed_prediction_examples.csv',index=False)
print('Transformer tahmini değişen örnek sayısı:',len(changed_ix))
print(pd.DataFrame(examples).head(3).to_string(index=False))

    emotion  support  baseline_drop  transformer_drop
       love      238       0.051477          0.053530
      anger      198       0.040396          0.047736
  gratitude      352       0.047266          0.047384
       fear       78       0.020216          0.041974
 admiration      504       0.038561          0.040497
     caring      135       0.010288          0.030813
   approval      351       0.012372          0.023188
  amusement      264       0.020401          0.019928
   surprise      141       0.009251          0.019863
        joy      161       0.023752          0.017665
    neutral     1787       0.003206          0.016830
disapproval      267       0.001275          0.016363
Transformer tahmini değişen örnek sayısı: 679
 index                                                                                                                                          original                                                                                                    

In [ ]:
def duygu_tahmin_et(metin):
    d=EmotionDataset([metin]);p=predict_transformer(model,DataLoader(d,batch_size=1))[0]
    labels=[(label_names[i],round(float(p[i]),3)) for i in np.where(p>=ESIK_TRANSFORMER)[0]]
    return sorted(labels,key=lambda x:-x[1])
demo_text='I am so happy and grateful for your help!'
print('Demo metni:',demo_text)
print('Tahmin:',duygu_tahmin_et(demo_text))

Demo metni: I am so happy and grateful for your help!


Transformer tahmin:   0%|          | 0/1 [00:00<?, ?it/s]

Tahmin: [('gratitude', 0.802), ('joy', 0.586)]


In [ ]:
import random
def set_seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## ADIM 5.3 — DistilBERT: 3 eğitim seed'i + ablasyon (%5 / %10 / %20)

**Neden?** Tek bir eğitim çalıştırması güvenilir değil: modelin ilk ağırlıkları ve verinin karıştırılma sırası rastgele. Aynı modeli 3 farklı **eğitim seed'iyle** (0, 1, 2) sıfırdan eğitip sonucun tekrar edip etmediğine bakıyoruz. Her seed için ADIM 8'deki ablasyon (%5, %10, %20 yazım hatası) da aynı yöntemle hesaplanır.

**Önemli:**
- Yazım hatalı test metinleri **değişmez**; sadece eğitim seed'i değişir. Baseline sonuçları (temiz 0.4473, seed 0 typo 0.4320) aynı kalır.
- Her seed'de model önceden eğitilmiş DistilBERT'ten **sıfırdan** yüklenir; optimizer ve öğrenme oranı planlayıcısı da yeniden kurulur.
- Yeniden başlatma gerekmez. Her seed bitince sonuç Drive'daki `GoEmotions_seed_runs` klasörüne kaydedilir; oturum düşerse hücreyi tekrar çalıştırınca biten seed'leri atlar.
- Mevcut deney hücrelerinin (ADIM 0–10) hiçbiri değiştirilmedi; bu yeni, ayrı bir hücre.

**Ön koşul:** ADIM 0–4.5 ve ADIM 5 hazırlık hücresi (TOKENIZER, EmotionDataset, train_ds, val_loader) bu oturumda çalışmış olmalı. ADIM 5.1–9'u çalıştırmana gerek yok.

**Süre:** T4'te seed başına yaklaşık 8–10 dakika, toplam yaklaşık 30 dakika.

In [30]:
# ADIM 5.3 — DistilBERT'i 3 farklı EĞİTİM seed'iyle sıfırdan eğit, değerlendir + ADIM 8 ablasyonu (%5/%10/%20)
# Ön koşul: ADIM 0–4.5 ve ADIM 5 hazırlık hücresi (TOKENIZER, EmotionDataset, train_ds, val_loader) çalışmış olmalı.
# Yazım hatalı test metinleri (noise_texts) SABİT kalır; sadece eğitim seed'i değişir.
# Her seed bitince sonuç bu Colab oturumundaki /content/GoEmotions_seed_runs klasörüne kaydedilir (Drive kullanılmaz).
import gc, json, math, time, copy
import numpy as np, pandas as pd, torch
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import set_seed, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SAVE_DIR = Path('/content/GoEmotions_seed_runs')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SEEDS = [0, 1, 2]
THRESHOLD_GRID = np.round(np.arange(0.10, 0.901, 0.05), 2)
BOOTSTRAP_N = 1000

@torch.no_grad()
def predict_probs(m, loader):
    m.eval(); out = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = m(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
        out.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.vstack(out)

@torch.no_grad()
def val_loss(m, loader):
    m.eval(); total, count = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        total += float(m(**batch).loss) * len(batch['input_ids']); count += len(batch['input_ids'])
    return total / count

def macro(y, p): return float(f1_score(y, p, average='macro', zero_division=0))

# Test verileri bir kez hazırlanır; her eğitim seed'inde AYNI metinler kullanılır
test_loader_orig = DataLoader(EmotionDataset(X_test), batch_size=EVAL_BATCH_SIZE)
noise_loaders = {g: DataLoader(EmotionDataset(noise_texts[g]), batch_size=EVAL_BATCH_SIZE) for g in NOISE_SEEDS}
# Ablasyon metinleri: ADIM 8 ile aynı yöntem (seed=i); %10 = ana deneydeki yazım hatalı test
ABLATION_RATES = [0.05, 0.10, 0.20]
abl_texts = {r: (noise_texts[0] if r == 0.10 else [yazim_hatasi_ekle(t, oran=r, seed=i)[0] for i, t in enumerate(X_test)]) for r in ABLATION_RATES}
abl_loaders = {r: DataLoader(EmotionDataset(abl_texts[r]), batch_size=EVAL_BATCH_SIZE) for r in ABLATION_RATES}
abl_base_pred = {r: (base_pred_typo if r == 0.10 else (baseline.predict_proba(tfidf.transform(abl_texts[r])) >= ESIK_BASELINE).astype(int)) for r in ABLATION_RATES}

def train_one_seed(seed):
    set_seed(seed)                                    # Python, NumPy, PyTorch, CUDA
    gen = torch.Generator(); gen.manual_seed(seed)    # eğitim verisinin karıştırılma sırası
    loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=gen)
    m = AutoModelForSequenceClassification.from_pretrained(      # her seed'de SIFIRDAN yükle
        MODEL_ID, num_labels=len(label_names), problem_type='multi_label_classification').to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total = math.ceil(len(loader) / GRAD_ACCUM) * TRAIN_EPOCHS
    sch = get_linear_schedule_with_warmup(opt, max(1, int(0.1 * total)), total)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    best, best_state, hist, t0 = float('inf'), None, [], time.time()
    for ep in range(TRAIN_EPOCHS):
        m.train(); opt.zero_grad(set_to_none=True); run = 0.0
        for step, batch in enumerate(tqdm(loader, desc=f'seed {seed} · epoch {ep+1}/{TRAIN_EPOCHS}')):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                loss = m(**batch).loss
            run += float(loss.detach())
            scaler.scale(loss / GRAD_ACCUM).backward()
            if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
        vl = val_loss(m, val_loader)
        hist.append({'epoch': ep + 1, 'train_loss': run / len(loader), 'val_loss': vl})
        print(f'  seed {seed} · epoch {ep+1}: train loss={run/len(loader):.4f}  val loss={vl:.4f}')
        if vl < best:
            best = vl; best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)
    return m, hist, time.time() - t0

def evaluate_seed(seed, m, hist, secs):
    # 1) Eşik SADECE doğrulama kümesinde seçilir
    pv = predict_probs(m, val_loader)
    val_scores = [(float(t), macro(y_val, (pv >= t).astype(int))) for t in THRESHOLD_GRID]
    thr, val_macro = max(val_scores, key=lambda x: x[1])
    # 2) Orijinal test ve 3 yazım hatası sürümü
    p_orig = (predict_probs(m, test_loader_orig) >= thr).astype(int)
    t_orig = macro(y_test, p_orig)
    per_noise, t_preds = [], {}
    for g in NOISE_SEEDS:
        t_preds[g] = (predict_probs(m, noise_loaders[g]) >= thr).astype(int)
        b_drop = base_original['macro_f1'] - macro(y_test, base_noise_preds[g])
        t_drop = t_orig - macro(y_test, t_preds[g])
        per_noise.append({'noise_seed': g, 'baseline_drop': b_drop, 'transformer_drop': t_drop, 'advantage': b_drop - t_drop})
    # 3) Önceden belirlenmiş ana test: yazım hatası seed 0 üzerinde eşleştirilmiş bootstrap
    rng = np.random.default_rng(42); n = len(y_test); adv = np.empty(BOOTSTRAP_N)
    for b in tqdm(range(BOOTSTRAP_N), desc=f'seed {seed} · bootstrap', leave=False):
        ix = rng.integers(0, n, size=n); yy = y_test[ix]
        d_base = macro(yy, base_pred_orig[ix]) - macro(yy, base_pred_typo[ix])
        d_tr = macro(yy, p_orig[ix]) - macro(yy, t_preds[0][ix])
        adv[b] = d_base - d_tr
    lo, hi = np.percentile(adv, [2.5, 97.5])
    # 4) Ablasyon: aynı model, %5 / %10 / %20 yazım hatası
    ablation = []
    for r in ABLATION_RATES:
        tp = t_preds[0] if r == 0.10 else (predict_probs(m, abl_loaders[r]) >= thr).astype(int)
        bd = base_original['macro_f1'] - macro(y_test, abl_base_pred[r]); td = t_orig - macro(y_test, tp)
        ablation.append({'typo_rate': r, 'baseline_drop': bd, 'transformer_drop': td, 'advantage': bd - td})
    res = {'train_seed': seed, 'training_seconds': secs, 'history': hist,
           'threshold': thr, 'val_macro_f1': val_macro,
           'test_original_macro_f1': t_orig,
           'test_typo_seed0_macro_f1': t_orig - per_noise[0]['transformer_drop'],
           'transformer_drop_seed0': per_noise[0]['transformer_drop'],
           'advantage_seed0': per_noise[0]['advantage'], 'bootstrap_ci95': [float(lo), float(hi)],
           'supported': bool(per_noise[0]['advantage'] > 0 and lo > 0), 'per_noise_seed': per_noise, 'ablation': ablation}
    (SAVE_DIR / f'distilbert_train_seed{seed}_v2.json').write_text(json.dumps(res, indent=2), encoding='utf-8')
    return res

results = {}
for seed in TRAIN_SEEDS:
    f = SAVE_DIR / f'distilbert_train_seed{seed}_v2.json'
    if f.exists():                                   # daha önce bittiyse tekrar eğitme
        results[seed] = json.loads(f.read_text()); print(f'seed {seed}: kayıttan yüklendi'); continue
    print(f'\n===== EĞİTİM SEED {seed} =====')
    m, hist, secs = train_one_seed(seed)
    results[seed] = evaluate_seed(seed, m, hist, secs)
    r = results[seed]
    print(f"seed {seed}: süre {secs/60:.1f} dk | eşik {r['threshold']} | temiz {r['test_original_macro_f1']:.4f} | "
          f"hatalı {r['test_typo_seed0_macro_f1']:.4f} | avantaj {r['advantage_seed0']*100:+.2f} puan | "
          f"GA [{r['bootstrap_ci95'][0]*100:.2f}, {r['bootstrap_ci95'][1]*100:.2f}]")
    del m; gc.collect(); torch.cuda.empty_cache()     # GPU belleğini boşalt

# ÖZET TABLO
summary_df = pd.DataFrame([{
    'train_seed': s, 'threshold': r['threshold'], 'clean_macro_f1': r['test_original_macro_f1'],
    'typo_macro_f1': r['test_typo_seed0_macro_f1'], 'transformer_drop': r['transformer_drop_seed0'],
    'advantage': r['advantage_seed0'], 'ci_low': r['bootstrap_ci95'][0], 'ci_high': r['bootstrap_ci95'][1],
    'supported': r['supported']} for s, r in results.items()])
print('\nBaseline: temiz', round(base_original['macro_f1'], 4), '| kayıp (seed 0)', round(base_drop, 4))
print(summary_df.round(4).to_string(index=False))
all_adv = [row['advantage'] for r in results.values() for row in r['per_noise_seed']]
print(f"\n3 eğitim seed'i ortalaması: temiz macro-F1 {summary_df.clean_macro_f1.mean():.4f} ± {summary_df.clean_macro_f1.std(ddof=1):.4f}, "
      f"avantaj {summary_df.advantage.mean()*100:+.2f} ± {summary_df.advantage.std(ddof=1)*100:.2f} puan")
print(f"3 eğitim × 3 yazım hatası = 9 karşılaştırmada avantaj: ortalama {np.mean(all_adv)*100:+.2f} puan, "
      f"pozitif olan {sum(a > 0 for a in all_adv)}/9")
summary_df.to_csv(SAVE_DIR / 'distilbert_three_train_seeds_summary.csv', index=False)
abl_df = pd.DataFrame([dict(train_seed=s, **a) for s, r in results.items() for a in r['ablation']])
abl_summary = abl_df.groupby('typo_rate')[['baseline_drop', 'transformer_drop', 'advantage']].agg(['mean', 'std'])
print('\nABLASYON (3 eğitim seed\'i, ortalama ± std; kayıplar macro-F1 puanı olarak):')
print((abl_summary * 100).round(2).to_string())
abl_df.to_csv(SAVE_DIR / 'ablation_three_train_seeds.csv', index=False)
print('\nKaydedildi:', SAVE_DIR)


===== EĞİTİM SEED 0 =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


seed 0 · epoch 1/2:   0%|          | 0/2714 [00:00<?, ?it/s]

  seed 0 · epoch 1: train loss=0.1633  val loss=0.0945


seed 0 · epoch 2/2:   0%|          | 0/2714 [00:00<?, ?it/s]

... (15 outputs left out)

seed 2 · epoch 2/2:   0%|          | 0/2714 [00:00<?, ?it/s]

  seed 2 · epoch 2: train loss=0.0856  val loss=0.0854


seed 2 · bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

seed 2: süre 6.1 dk | eşik 0.2 | temiz 0.4473 | hatalı 0.4315 | avantaj -0.05 puan | GA [-0.97, 0.90]

Baseline: temiz 0.4473 | kayıp (seed 0) 0.0153
 train_seed  threshold  clean_macro_f1  typo_macro_f1  transformer_drop  advantage  ci_low  ci_high  supported
          0       0.15          0.4249         0.4104            0.0144     0.0009 -0.0078   0.0084      False
          1       0.20          0.4464         0.4311            0.0153     0.0000 -0.0095   0.0087      False
          2       0.20          0.4473         0.4315            0.0158    -0.0005 -0.0097   0.0090      False

3 eğitim seed'i ortalaması: temiz macro-F1 0.4395 ± 0.0127, avantaj +0.01 ± 0.07 puan
3 eğitim × 3 yazım hatası = 9 karşılaştırmada avantaj: ortalama +0.49 puan, pozitif olan 8/9

ABLASYON (3 eğitim seed'i, ortalama ± std; kayıplar macro-F1 puanı olarak):
          baseline_drop      transformer_drop       advantage      
                   mean  std             mean   std      mean   std
typo_rate    

## ADIM 10 — Sonuç tabloları, grafikler ve yeniden üretilebilir rapor

**Neden?** Deneyin bütün gerçek sayısal çıktıları JSON, CSV ve PNG olarak saklanır. Sonuçlar henüz çalıştırılmadan rapora yazılmaz. **Form:** Kutu 5–7.

In [ ]:
import matplotlib.pyplot as plt
summary={
 'dataset':{'train':len(X_train),'validation':len(X_val),'test':len(X_test),'labels':len(label_names),'license':'Apache-2.0'},
 'typo_generator':{'eligible_words':int(noise_df.iloc[0]['eligible_words']),'modified_words':int(noise_df.iloc[0]['changed_words']),'observed_rate':float(noise_df.iloc[0]['realized_rate']),'comments_modified':int(noise_df.iloc[0]['changed_comments'])},
 'baseline':baseline_results,
 'transformer':{'model_id':MODEL_ID,'epochs':TRAIN_EPOCHS,'training_seconds':training_seconds,'threshold':ESIK_TRANSFORMER,'original':trans_original,'typo_10pct':trans_typo,'macro_f1_drop':trans_drop},
 'always_neutral_reference':neutral_reference,
 'three_noise_seeds':replicate_rows,
 'comparison':{'observed_advantage_baseline_drop_minus_transformer_drop':observed_advantage,'bootstrap_iterations':BOOTSTRAP_N,'bootstrap_ci95':[float(ci_low),float(ci_high)],'transformer_more_robust_supported':bool(supported)}
}
(OUTPUT_DIR/'experiment_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
comparison_df=pd.DataFrame([
 {'model':'TF-IDF + Logistic Regression','original_macro_f1':base_original['macro_f1'],'typo_macro_f1':base_typo['macro_f1'],'macro_f1_drop':base_drop},
 {'model':'Fine-tuned DistilBERT','original_macro_f1':trans_original['macro_f1'],'typo_macro_f1':trans_typo['macro_f1'],'macro_f1_drop':trans_drop}
])
comparison_df.to_csv(OUTPUT_DIR/'model_comparison.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4.5))
xx=np.arange(2);w=0.34
ax.bar(xx-w/2,comparison_df.original_macro_f1,width=w,label='Original')
ax.bar(xx+w/2,comparison_df.typo_macro_f1,width=w,label='10% typos')
ax.set_xticks(xx,comparison_df.model,rotation=0);ax.set_ylabel('Macro-F1');ax.set_ylim(0,1);ax.legend();ax.set_title('GoEmotions: original vs typo test')
fig.tight_layout();fig.savefig(OUTPUT_DIR/'macro_f1_comparison.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(8,4.5))
ax.plot(ablation_df.typo_probability,ablation_df.baseline_drop,marker='o',label='TF-IDF + LR')
ax.plot(ablation_df.typo_probability,ablation_df.transformer_drop,marker='o',label='DistilBERT')
ax.set_xlabel('Typo probability per eligible word');ax.set_ylabel('Macro-F1 drop');ax.legend();ax.set_title('Ablation: noise severity')
fig.tight_layout();fig.savefig(OUTPUT_DIR/'typo_ablation.png',dpi=160);plt.show()
print(comparison_df.to_string(index=False))

## ADIM 11 — Güncel literatür (2025–2026) ve araştırma boşluğu

**Veri seti:** Demszky et al. (2020), *GoEmotions: A Dataset of Fine-Grained Emotions*, ACL. https://arxiv.org/abs/2005.00547

**Graduate related work #1:** Alqarni et al. (2025), *Emotion-Aware RoBERTa enhanced with emotion-specific attention and TF-IDF gating for fine-grained emotion recognition*, **Scientific Reports**, https://doi.org/10.1038/s41598-025-99515-6 — emotion classification with a Transformer architecture. **Difference:** our outcome is the *drop* in multi-label GoEmotions macro-F1 under fixed character-level typos, compared to a sparse baseline.

**Graduate related work #2:** Gao et al. (2026), *A diffusion-inspired noise augmentation framework for robust multilingual text classification*, **Scientific Reports**, https://www.nature.com/articles/s41598-026-59500-z — robustness through noise augmentation. **Difference:** our project does **not** propose augmentation; it directly measures the comparative degradation of two model families on the same paired GoEmotions comments.

**Supporting:** Moffett & Dhingra (2025), *Close or Cloze? Assessing the Robustness of Large Language Models to Adversarial Perturbations via Word Recovery*, **COLING 2025**, https://aclanthology.org/2025.coling-main.467/ — textual perturbations and word recovery. **Difference:** our task is fine-grained multi-label emotion prediction rather than word recovery.

**Gap, without overstating novelty:** These studies address emotion modelling and perturbation robustness, but do not themselves establish the paired macro-F1 degradation comparison between TF-IDF + Logistic Regression and fine-tuned DistilBERT on the simplified GoEmotions split using this specific typo protocol. This is a targeted comparison, not a claim that nobody has studied emotion robustness.

**Ablations:** 5%, 10%, 20% eligible-word typo probability; three independent perturbation seeds; per-emotion F1 and manual inspection of examples. **Limitations:** synthetic errors may change meaning; comments may already contain typos; model capacities differ; class imbalance and compute constraints.

**Before submission:** verify author spelling and publication dates from linked publisher pages, and fill real member information. Source links are provided for auditability.

## ADIM 12 — EN SON: Hocanın orijinal 2 sayfalık formu ve 5 slaytlık sunum

**ŞİMDİ ÇALIŞTIRMA.** Önce ADIM 0–11 tamamlanmalı: baseline, DistilBERT, test karşılaştırması, paired bootstrap, %5/%10/%20 ablasyon, hata analizi ve gerçek sonuç dosyaları.

Bu adım **yalnızca teslim materyallerini** oluşturur:
- `TeamXX_Proposal.pdf`: hocanın orijinal iki sayfalık formuna araştırma sorusu, yöntem, **gerçek ölçülen sonuçlar**, akademik literatür ve takım görevleri yazılır.
- `TeamXX_LightningTalk.pptx`: en fazla 5 slayt, 5 dakikalık İngilizce lightning talk.

Varsayılan `GENERATE_DELIVERABLES = False` olduğundan bu iki kod hücresi **Run all sırasında otomatik atlanır**. Deneyler bittikten, isimler ve takım numarası girildikten sonra ADIM 0'daki bayrağı `True` yapıp **sadece teslim hücrelerini** çalıştır. Bu bayrağı değiştirmek için ADIM 0'ı yeniden çalıştırmak, Colab oturumundaki eğitilmiş modeli sıfırlamaz; ancak bütün notebook'u yeniden çalıştırma.

**Dikkat:** Formun içerikleri son kez gözle kontrol edilmeli; model sonuçları ve makale atıfları otomatik oluşturulsa da insan doğrulaması zorunludur.


In [ ]:
if not GENERATE_DELIVERABLES:
    print("ADIM 12/13 ertelendi: deneyleri tamamladıktan sonra GENERATE_DELIVERABLES = True yapın.")
else:
    # ADIM 12 — HOCANIN ORİJİNAL İKİ SAYFALIK FORMUNU DOLDUR
    # Template PDF is embedded so this Colab notebook works without an extra file.
    import fitz, base64, zlib, textwrap
    from pathlib import Path
    TEMPLATE_COMPRESSED_B64 = "..."
    template_bytes=zlib.decompress(base64.b64decode(TEMPLATE_COMPRESSED_B64))
    source_pdf=fitz.open(stream=template_bytes,filetype='pdf')
    proposal_pdf=fitz.open(); proposal_pdf.insert_pdf(source_pdf,from_page=2,to_page=3)
    proposal_path=OUTPUT_DIR/f'Team{TEAM_NUMBER}_Proposal.pdf'

    # The professor's form labels/instructions remain unchanged; insert text into their blank areas.
    def fill(page, rect, content, font=8.2, min_font=6.7):
        if isinstance(content,(list,tuple)): content='\n'.join(map(str,content))
        box=fitz.Rect(*rect)
        size=font
        while size>=min_font:
            # negative return means the text did not fit; insert_textbox doesn't draw in that case
            remaining=page.insert_textbox(box,str(content),fontname='helv',fontsize=size,lineheight=1.17,color=(0.08,0.18,0.30))
            if remaining>=0: return
            size-=0.2
        raise ValueError(f'Text does not fit in form area {rect}. Shorten content rather than overflowing.')

    p1,p2=proposal_pdf[0],proposal_pdf[1]
    fill(p1,(225,77,548,102),f'{TEAM_NAME} | Team {TEAM_NUMBER}',9)
    fill(p1,(225,106,551,167),'\n'.join(MEMBERS),7.8)
    # Mark the selected project type in the original form without changing its labels.
    # Proje tipi: "Model" kutusuna tik (formun kendi ☐ işaretinin üstüne)
    p1.draw_line(fitz.Point(210.3,178.5),fitz.Point(212.8,181.6),color=(0.05,0.3,0.55),width=1.4)
    p1.draw_line(fitz.Point(212.8,181.6),fitz.Point(216.2,173.6),color=(0.05,0.3,0.55),width=1.4)

    # 1 · Research question
    fill(p1,(50,236,546,308),
     'On the GoEmotions test set (5,427 Reddit comments, 28 emotion labels), when about 10% of eligible words (4+ letters; '
     '[NAME]/[RELIGION] masks excluded) receive synthetic character-level typos (delete / repeat / swap adjacent letters), '
     'does a fine-tuned DistilBERT lose less macro-F1 than a TF-IDF + one-vs-rest logistic regression baseline, '
     'relative to the original comments?\n'
     'The answer is NO if the observed advantage (baseline drop minus DistilBERT drop) is <= 0, or if its paired-bootstrap '
     '95% confidence interval includes zero.',8.0)

    # 2 · Why it matters
    fill(p1,(50,344,546,423),
     'Emotion classifiers are used on informal user text (social media monitoring, support tickets, moderation), '
     'where typos are common. Practitioners need to know whether a cheap sparse baseline or a fine-tuned Transformer '
     'degrades more gracefully. Closest work: GoEmotions (Demszky et al., 2020) reports clean-text results only '
     '(BERT macro-F1 0.46); Moffett & Dhingra (COLING 2025) and Gao et al. (Sci. Rep. 2026) study robustness to '
     'perturbations but not fine-grained multi-label emotion under a controlled character-typo rate.',7.8)

    # 3 · Data
    fill(p1,(50,472,546,566),
     f'GoEmotions (Demszky et al., 2020), "simplified" config; source: '
     f'https://huggingface.co/datasets/google-research-datasets/go_emotions; licence: Apache-2.0; language: English (Reddit). '
     f'Loaded train / dev / test: {len(X_train):,} / {len(X_val):,} / {len(X_test):,}; 27 emotions + neutral, multi-label.\n'
     f'Real example (verbatim, train): "{REAL_EXAMPLE_TEXT}" -> {REAL_EXAMPLE_LABEL}',7.8)

    # 4 · Method and baseline
    fill(p1,(50,614,546,781),
     'Baseline: TF-IDF (20k uni+bigram features) + one-vs-rest logistic regression (class-balanced). '
     'Method: fine-tune distilbert-base-uncased with 28 sigmoid outputs and BCE loss (2 epochs, LR 3e-5, max 128 tokens). '
     f'Compute: one Colab T4 GPU; one fine-tuning run took about {training_seconds/60:.0f} minutes, so 3 training seeds '
     'plus 3 typo rates fit in a few GPU hours.\n'
     'Fair comparison: both models are trained on the same original training split, use a single decision threshold '
     'chosen on the validation set only, and are evaluated on exactly the same original and perturbed test comments. '
     'The typo generator is seeded and never changes the first letter of a word.',8.0)

    # 5 · Evaluation
    fill(p2,(50,80,546,156),
     'Metric: macro-F1 (each of the 28 emotions counts equally); micro-F1 reported as a secondary metric. '
     'Drop = macro-F1(original) - macro-F1(perturbed), on the same 5,427 test comments. '
     'Real difference: 1,000 paired bootstrap resamples of the test set; we call the advantage real only if the 95% CI '
     'of (baseline drop - DistilBERT drop) lies entirely above zero. Noise check: 3 typo seeds; 3 DistilBERT training seeds.',7.9)

    # 6 · First look
    fill(p2,(50,194,546,308),
     f'Executed runs (10% typos, noise seed 0). TF-IDF + LR: macro-F1 {base_original["macro_f1"]:.3f} -> '
     f'{base_typo["macro_f1"]:.3f}, drop {base_drop*100:.2f} points (mean over 3 typo seeds: '
     f'{noise_df.baseline_drop.mean()*100:.2f} points). DistilBERT (preliminary, one training run): '
     f'{trans_original["macro_f1"]:.3f} -> {trans_typo["macro_f1"]:.3f}, drop {trans_drop*100:.2f} points. '
     f'Advantage {observed_advantage*100:.2f} points, paired-bootstrap 95% CI [{ci_low*100:.2f}, {ci_high*100:.2f}].\n'
     'What it shows: synthetic typos measurably lower macro-F1, and the size of the gap between the two models. '
     'What it does not show: whether the result is stable across DistilBERT training seeds, other typo rates, '
     'or real (non-synthetic) typos.',7.9)

    # 7 · Research lead section
    fill(p2,(50,356,546,470),
     f'Research lead: {MEMBERS[0]}.\n'
     'Moffett & Dhingra (COLING 2025) benchmark nine perturbation types, including character typos, and show that LLMs '
     'struggle to recover perturbed words, which hurts toxic-content classification. Gao et al. (Sci. Rep. 16:29779, 2026) '
     'propose NoiseDiffuser, a noise-augmentation method that makes TF-IDF classifiers and BERT more robust on topic '
     'classification; their perturbations are word-level. Alqarni et al. (Sci. Rep. 15:17617, 2025) improve emotion '
     'recognition with an enhanced RoBERTa. Difference: none of them measures how much a sparse baseline and a fine-tuned '
     'Transformer each lose under the same controlled character-typo protocol on 28-label multi-label emotion data.\n'
     'Planned ablation: typo rates 5 / 10 / 20%, 3 training seeds, per-emotion analysis; published baseline: compare our '
     'clean macro-F1 with GoEmotions BERT (0.46).',7.5)

    # 8 · Biggest risk
    fill(p2,(50,507,546,568),
     'Risk 1: the difference is too small to separate from noise. Fallback: report it as a negative result with its CI, '
     'and use the 20% typo rate to show the trend. Risk 2: GPU quota. Fallback: fewer training seeds, disclosed. '
     'Risk 3: rare emotions (e.g. grief, 6 test examples) give unstable per-label F1; we label those results exploratory.',7.8)

    # 9 · Plan and roles
    fill(p2,(50,615,546,694),
     'Mid-project: baseline and DistilBERT numbers on original and 10% typo test, 3 typo seeds, bootstrap CI. '
     'Final: 3 training seeds, 5/10/20% ablation, per-emotion error analysis, noise-augmented training as an extension, code.\n'
     f'Roles: {TEAM_ROLES}.\n'
     f'AI tools: {AI_TOOLS_USED} were used for planning, code and wording; all numbers come from our own executed runs.',7.6)

    # 10 · Beyond the course
    fill(p2,(50,732,546,786),
     'Possibly a short workshop paper on robustness of emotion classifiers to real-world noise (e.g. naturally occurring '
     'typos, Turkish emotion data). Not yet validated.',8.0)

    proposal_pdf.save(str(proposal_path),garbage=4,deflate=True)
    assert len(fitz.open(str(proposal_path)))==2
    print('Hocanın orijinal formu dolduruldu:',proposal_path)


In [ ]:
if not GENERATE_DELIVERABLES:
    print("ADIM 12/13 ertelendi: deneyleri tamamladıktan sonra GENERATE_DELIVERABLES = True yapın.")
else:
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from pptx.dml.color import RGBColor
    prs=Presentation();prs.slide_width=Inches(13.33);prs.slide_height=Inches(7.5)
    def slide(title,bullets):
        s=prs.slides.add_slide(prs.slide_layouts[6]);
        bg=s.background.fill;bg.solid();bg.fore_color.rgb=RGBColor(247,249,252)
        tb=s.shapes.add_textbox(Inches(.75),Inches(.55),Inches(11.8),Inches(.8));p=tb.text_frame.paragraphs[0];p.text=title;p.font.size=Pt(30);p.font.bold=True;p.font.color.rgb=RGBColor(23,47,80)
        body=s.shapes.add_textbox(Inches(.85),Inches(1.65),Inches(11.7),Inches(5.25));tf=body.text_frame;tf.word_wrap=True
        for j,line in enumerate(bullets):
            p=tf.paragraphs[0] if j==0 else tf.add_paragraph();p.text='• '+str(line);p.font.size=Pt(22 if len(str(line))<115 else 19);p.space_after=Pt(20);p.font.color.rgb=RGBColor(37,51,67)
        return s
    slide('Research question',[
     'Does fine-tuned DistilBERT lose less macro-F1 than TF-IDF + Logistic Regression under 10% synthetic typos?',
     'No if the paired-bootstrap 95% CI includes zero or the observed advantage is non-positive.',
     'Task: 28-label GoEmotions emotion classification'])
    slide('Data and controlled perturbation',[
     'English GoEmotions: 43,410 train / 5,426 validation / 5,427 test; Apache-2.0.',
     f"{int(noise_df.iloc[0]['changed_words'])} / {int(noise_df.iloc[0]['eligible_words'])} eligible words changed ({100*noise_df.iloc[0]['realized_rate']:.2f}%); {int(noise_df.iloc[0]['changed_comments'])} comments affected.",
     'Delete / repeat / swap letters; protect [NAME] and similar placeholders.'])
    slide('Models and fair evaluation',[
     'Baseline: TF-IDF (20k features) + One-vs-Rest Logistic Regression.',
     'DistilBERT: fine-tuned 2 epochs; multi-label BCE objective.',
     'Same test comments; original vs typo; each model threshold fixed from validation.'])
    slide('Measured results',[
     f'Baseline macro-F1: {base_original["macro_f1"]:.4f} → {base_typo["macro_f1"]:.4f}; drop {base_drop:.4f}.',
     f'DistilBERT macro-F1: {trans_original["macro_f1"]:.4f} → {trans_typo["macro_f1"]:.4f}; drop {trans_drop:.4f}.',
     f'Paired advantage: {observed_advantage:.4f}; 95% CI [{ci_low:.4f}, {ci_high:.4f}].',
     'Conclusion: '+('supports smaller Transformer loss.' if supported else 'does NOT establish smaller Transformer loss.')])
    slide('Research contribution, risks, next steps',[
     'Recent papers: Alqarni et al. (2025); Gao et al. (2026); Moffett & Dhingra (2025).',
     'Ablation: 5%, 10%, 20% typo rates; per-emotion errors and runnable demo.',
     'Limits: synthetic noise, label imbalance, 3 noise seeds, compute differences.',
     'AI disclosure: '+AI_TOOLS_USED+' assisted with planning, code and drafting.'])
    pptx_path=OUTPUT_DIR/f'Team{TEAM_NUMBER}_LightningTalk.pptx';prs.save(pptx_path)
    print('PPTX:',pptx_path)

## ADIM 13 — EN SON: Teslim dosyalarını ZIP olarak indir

**ŞİMDİ ÇALIŞTIRMA.** Bu adım deney yürütmez, model eğitmez ve sonuç üretmez. ADIM 12'deki **orijinal form PDF'i** ve **5 slaytlık sunum** hazırlandıktan sonra, bunları ve araştırma sonuçlarını ZIP'e koyar. DistilBERT'in büyük ağırlık dosyaları ZIP'e alınmaz.

Önce `TEAM_NUMBER`, `MEMBERS`, `TEAM_ROLES`, kullanılan AI araçları, PDF'nin **2 sayfa** ve sunumun **5 slayt** olduğunu kontrol et. Her takım üyesi proposal'ı kendi hesabından teslim eder. `GENERATE_DELIVERABLES = False` iken ZIP kodu otomatik atlanır.


In [ ]:
import zipfile
from reportlab.pdfbase.pdfmetrics import stringWidth

In [ ]:
if not GENERATE_DELIVERABLES:
    print("ADIM 12/13 ertelendi: deneyleri tamamladıktan sonra GENERATE_DELIVERABLES = True yapın.")
else:
    import zipfile
    zip_path=Path('/content')/f'Team{TEAM_NUMBER}_GoEmotions_Project.zip'
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in OUTPUT_DIR.rglob('*'):
            if f.is_file() and 'distilbert_goemotions_model' not in f.parts:
                z.write(f,arcname=f.relative_to(OUTPUT_DIR))
    print('Hazır ZIP:',zip_path,'| Boyut MB:',round(zip_path.stat().st_size/1e6,2))
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print('Colab dışı ortam: ZIP yerel diskte hazır.')

## Çalışma tamamlandığında kontrol listesi

### A. ÖNCE — Araştırma ve gerçek deneyler (ADIM 0–11)
- [ ] Veri seti ve eğitim/doğrulama/test ayrımı kontrol edildi.
- [ ] Baseline eğitildi; orijinal ve %10 yazım hatalı test Macro-F1/Micro-F1 hesaplandı.
- [ ] DistilBERT fine-tuning tamamlandı; aynı test çiftinde iki performans hesaplandı.
- [ ] Eşikler yalnızca doğrulama kümesinde seçildi; test için değiştirilmedi.
- [ ] İki modelin Macro-F1 kayıpları karşılaştırıldı; paired bootstrap %95 güven aralığı ve hipotez yorumu raporlandı.
- [ ] Üç gürültü tohumu, %5/%10/%20 ablasyon ve duygu bazlı hata analizi tamamlandı.
- [ ] Gerçek deney çıktıları CSV/JSON/grafik olarak kaydedildi; sayılar uydurulmadı.

### B. EN SON — ADIM 12: Orijinal form PDF'i ve sunum
- [ ] `GENERATE_DELIVERABLES = True` yapıldı; yalnızca teslim hücreleri çalıştırıldı.
- [ ] Hocanın **orijinal formu** gerçek bulgularla dolduruldu, `TeamXX_Proposal.pdf` **2 sayfa** olarak kontrol edildi.
- [ ] Güncel makaleler, gerçek takım bilgileri, görev dağılımı ve kullanılan AI araçları doğrulandı.
- [ ] `TeamXX_LightningTalk.pptx` **5 slayt** ve sunum provası **5 dakikayı aşmıyor**.

### C. EN SON — ADIM 13: ZIP ve teslim
- [ ] PDF, PPTX ve deney çıktıları kontrol edilip ZIP'e alındı; büyük model ağırlıkları hariç tutuldu.
- [ ] Takımın her üyesi proposal'ı kendi hesabından yükledi.

**Çalışma sırası:** Baseline → DistilBERT → test karşılaştırması → bootstrap → ablasyon → hata analizi → rapor → **ADIM 12 (PDF + sunum)** → **ADIM 13 (ZIP + teslim)**.

**Not:** Bu araştırma yalnızca demo değildir; gerçek sonuçlar ancak Colab'da kodun çalıştırılmasıyla elde edilir. Notebook dosyasında çıktı bulunmaması, çalışmanın hiç yapılmadığını tek başına kanıtlamaz; Colab'daki oturum sonuçları ayrıca doğrulanmalıdır.
